# Guardrails Gateway — MAD Pipeline (GPU Notebook)
**SJSU CS298B · Team 3 · 2025-26**

Multi-Agent Debate (MAD) output guardrail — fully self-contained, no Ollama.

| Component | Backend |
|-----------|----------|
| RAG embedder | `nvidia/llama-embed-nemotron-8b` on GPU (HuggingFace) |
| Agent A + Agent B + Claim Extractor | `Qwen/Qwen2.5-7B-Instruct` on GPU (HuggingFace) |
| Judge | `claude-haiku-4-5-20251001` (Anthropic API) |
| Vector DB | Qdrant Cloud (4,662 regulatory chunks) |

### VRAM requirements
- **≥ 32 GB** (A100, H100): both models load at bfloat16, fastest
- **18–32 GB** (RTX 3090, V100 32 GB): bfloat16 — may need to load sequentially
- **< 18 GB** (T4 16 GB): Qwen-7B auto-loaded in 4-bit NF4 (~4 GB), Nemotron-8B at bfloat16
- **Tight on T4**: change `AGENT_MODEL` to `Qwen/Qwen2.5-3B-Instruct` (~6 GB bfloat16)


In [ ]:
# Install all required packages
!pip install -q openai anthropic pydantic qdrant-client
!pip install -q transformers accelerate einops sentencepiece
!pip install -q bitsandbytes   # needed for 4-bit quantisation on GPUs < 18 GB VRAM
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
print('\u2713 All dependencies installed')


In [ ]:
import torch

print('=== GPU / Device Info ===')
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    n = torch.cuda.device_count()
    for i in range(n):
        props = torch.cuda.get_device_properties(i)
        vram  = props.total_memory / 1e9
        print(f'  GPU {i}  : {props.name}  {vram:.1f} GB VRAM')
        if vram >= 32:
            print(f'         -> bfloat16 for both models (no quantisation needed)')
        elif vram >= 18:
            print(f'         -> bfloat16 for Qwen-7B, bfloat16 for Nemotron-8B')
        else:
            print(f'         -> 4-bit NF4 for Qwen-7B + bfloat16 for Nemotron-8B')
else:
    print('  No CUDA GPU — running on CPU (very slow)')


In [ ]:
import os

# ── Qdrant vector database ─────────────────────────────────────────────────
os.environ['QDRANT_URL']        = 'https://e2e7b7d2-4927-4c61-a78d-61f9c4e024bb.us-east4-0.gcp.cloud.qdrant.io'
os.environ['QDRANT_API_KEY']    = 'QDRANT_API_KEY_REDACTED'
os.environ['QDRANT_COLLECTION'] = 'ai_governance_chunks_nemotron8b'
os.environ['QDRANT_TIMEOUT']    = '60'

# ── Embedding model (Nemotron-8B) ──────────────────────────────────────────
# Set to a local path if already downloaded, e.g. '/workspace/nemotron_download'
os.environ['EMBED_MODEL'] = 'nvidia/llama-embed-nemotron-8b'
os.environ['HF_TOKEN']    = 'HF_TOKEN_REDACTED'

# ── Agent LLM (HuggingFace — no Ollama needed) ────────────────────────────
# Qwen2.5-7B-Instruct: best quality, auto 4-bit on < 18 GB VRAM
# Alternative: Qwen/Qwen2.5-3B-Instruct  (smaller, faster, lower VRAM)
os.environ['AGENT_MODEL'] = 'Qwen/Qwen2.5-7B-Instruct'

# ── Judge (Claude Haiku via Anthropic API) ─────────────────────────────────
os.environ['JUDGE_PROVIDER']    = 'anthropic'
os.environ['JUDGE_MODEL']       = 'claude-haiku-4-5-20251001'
os.environ['ANTHROPIC_API_KEY'] = 'ANTHROPIC_API_KEY_REDACTED'

# ── Debate settings ────────────────────────────────────────────────────────
os.environ['MAX_CYCLES']                = '2'
os.environ['TOP_K_CHUNKS']              = '5'
os.environ['TOP_K_CHALLENGE_CHUNKS']    = '3'
os.environ['CONFIDENCE_THRESHOLD_HIGH'] = '0.8'
os.environ['CONFIDENCE_THRESHOLD_LOW']  = '0.4'

# ── Storage (SQLite) ───────────────────────────────────────────────────────
os.environ['MAD_DB_PATH'] = '/content/mad_store.db'

print('\u2713 Environment configured')


In [ ]:
import base64, os, sys

# All pipeline modules encoded as base64 — decoded and written to /content/
MODULES = {
    "rag/__init__.py": "",
    "rag/config.py": "IiIiCnJhZy9jb25maWcucHkg4oCUIGNvbmZpZ3VyYXRpb24gZm9yIHRoZSByZWFsIFFkcmFudCBSQUcgcmV0cmlldmVyLgoKQWxsIHZhbHVlcyBhcmUgb3ZlcnJpZGFibGUgdmlhIGVudmlyb25tZW50IHZhcmlhYmxlcy4KUWRyYW50IFVSTCBhbmQgQVBJIGtleSBtdXN0IGJlIHNldCDigJQgZWl0aGVyIGhlcmUgb3IgYXMgZW52IHZhcnMuCkRvIE5PVCBjb21taXQgcmVhbCBBUEkga2V5cyB0byBnaXQuIFVzZSBlbnZpcm9ubWVudCB2YXJpYWJsZXMgaW4gcHJvZHVjdGlvbi4KIiIiCmltcG9ydCBvcwoKIyDilIDilIAgUWRyYW50IGNvbm5lY3Rpb24g4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSAClFEUkFOVF9VUkw6IHN0ciA9IG9zLmdldGVudigKICAgICJRRFJBTlRfVVJMIiwKICAgICJodHRwczovL2UyZTdiN2QyLTQ5MjctNGM2MS1hNzhkLTYxZjljNGUwMjRiYi51cy1lYXN0NC0wLmdjcC5jbG91ZC5xZHJhbnQuaW8iCikKUURSQU5UX0FQSV9LRVk6IHN0ciA9IG9zLmdldGVudigKICAgICJRRFJBTlRfQVBJX0tFWSIsCiAgICAiIiAgICMgc2V0IHZpYSBlbnYgdmFyIOKAlCBuZXZlciBoYXJkY29kZSBpbiBwcm9kdWN0aW9uCikKUURSQU5UX1RJTUVPVVQ6IGludCA9IGludChvcy5nZXRlbnYoIlFEUkFOVF9USU1FT1VUIiwgIjYwIikpCgojIOKUgOKUgCBDb2xsZWN0aW9uIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgApDT0xMRUNUSU9OX05BTUU6IHN0ciA9IG9zLmdldGVudigKICAgICJRRFJBTlRfQ09MTEVDVElPTiIsCiAgICAiYWlfZ292ZXJuYW5jZV9jaHVua3NfbmVtb3Ryb244YiIKKQpWRUNUT1JfU0laRTogaW50ID0gNDA5NiAgICMgbnZpZGlhL2xsYW1hLWVtYmVkLW5lbW90cm9uLThiIG91dHB1dCBkaW1lbnNpb24KCiMg4pSA4pSAIEVtYmVkZGluZyBtb2RlbCDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKRU1CRURfTU9ERUw6IHN0ciA9IG9zLmdldGVudigKICAgICJFTUJFRF9NT0RFTCIsCiAgICAibnZpZGlhL2xsYW1hLWVtYmVkLW5lbW90cm9uLThiIgopCkVNQkVEX01BWF9MRU5HVEg6IGludCA9IDUxMiAgICMgbWF4IHRva2VucyBwZXIgY2h1bmsvcXVlcnkg4oCUIG1hdGNoZXMgaW5nZXN0CgojIFF1ZXJ5IHByZWZpeCByZXF1aXJlZCBieSB0aGUgTmVtb3Ryb24gaW5zdHJ1Y3Rpb24tZm9sbG93aW5nIGVtYmVkZGluZyBtb2RlbC4KIyBUaGlzIE1VU1QgYmUgcHJlcGVuZGVkIHRvIGV2ZXJ5IHF1ZXJ5IGF0IHJldHJpZXZhbCB0aW1lLgojIENodW5rcyB3ZXJlIGluZ2VzdGVkIFdJVEhPVVQgdGhpcyBwcmVmaXggKG9ubHkgcXVlcmllcyB1c2UgaXQpLgpRVUVSWV9QUkVGSVg6IHN0ciA9ICgKICAgICJJbnN0cnVjdDogUmV0cmlldmUgcmVsZXZhbnQgcmVndWxhdG9yeSBwYXNzYWdlIHRvIGFuc3dlciB0aGUgcXVlcnlcblF1ZXJ5OiAiCikKCiMgSHVnZ2luZ0ZhY2UgdG9rZW4g4oCUIHJlcXVpcmVkIGlmIG1vZGVsIGlzIGdhdGVkCkhGX1RPS0VOOiBzdHIgPSBvcy5nZXRlbnYoIkhGX1RPS0VOIiwgIiIpCg==",
    "rag/embedder.py": "IiIiCnJhZy9lbWJlZGRlci5weSDigJQgTmVtb3Ryb24tOEIgZW1iZWRkaW5nIG1vZGVsIHdyYXBwZXIuCj09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KCkxvYWRzIG52aWRpYS9sbGFtYS1lbWJlZC1uZW1vdHJvbi04YiBvbmNlIChzaW5nbGV0b24pIGFuZCBleHBvc2VzCnR3byBmdW5jdGlvbnM6CgogICAgZW1iZWRfcXVlcnkodGV4dCkgICAgICAgIOKAlCBwcmVwZW5kcyBRVUVSWV9QUkVGSVgsIHJldHVybnMgTGlzdFtmbG9hdF0KICAgIGVtYmVkX3RleHRzKHRleHRzKSAgICAgICDigJQgYmF0Y2ggZW1iZWQgV0lUSE9VVCBwcmVmaXggKGZvciBpbmdlc3QpCgpUaGUgZGlzdGluY3Rpb24gYmV0d2VlbiBlbWJlZF9xdWVyeSBhbmQgZW1iZWRfdGV4dHMgaXMgY3JpdGljYWw6CiAgLSBBdCBJTkdFU1QgdGltZTogY2h1bmtzIHdlcmUgZW1iZWRkZWQgd2l0aG91dCBhbnkgcHJlZml4IChyYXcgdGV4dCBvbmx5KQogIC0gQXQgUVVFUlkgdGltZTogIHF1ZXJpZXMgbXVzdCBiZSBwcmVmaXhlZCB3aXRoIFFVRVJZX1BSRUZJWCBzbyB0aGUgbW9kZWwKICAgICAgICAgICAgICAgICAgICBtYXBzIHRoZSBxdWVyeSBpbnRvIHRoZSBzYW1lIHZlY3RvciBzcGFjZSBhcyB0aGUgY2h1bmtzCgpUaGlzIG1hdGNoZXMgZXhhY3RseSB3aGF0IHRoZSBub3RlYm9vayBkb2VzIGluIGNlbGxzIDEwIGFuZCAxNi4KClRoZSBtb2RlbCBpcyBsb2FkZWQgbGF6aWx5IG9uIGZpcnN0IGNhbGwg4oCUIHNvIGltcG9ydGluZyB0aGlzIG1vZHVsZQpkb2VzIG5vdCB0cmlnZ2VyIGEgR1BVIGxvYWQgdW50aWwgcmV0cmlldmUoKSBpcyBhY3R1YWxseSBjYWxsZWQuCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgb3MKZnJvbSB0eXBpbmcgaW1wb3J0IExpc3QsIE9wdGlvbmFsCgppbXBvcnQgdG9yY2gKZnJvbSB0cmFuc2Zvcm1lcnMgaW1wb3J0IEF1dG9Ub2tlbml6ZXIsIEF1dG9Nb2RlbAoKZnJvbSByYWcuY29uZmlnIGltcG9ydCBFTUJFRF9NT0RFTCwgRU1CRURfTUFYX0xFTkdUSCwgUVVFUllfUFJFRklYLCBIRl9UT0tFTgoKIyDilIDilIAgU2luZ2xldG9uIHN0YXRlIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgApfdG9rZW5pemVyID0gTm9uZQpfbW9kZWwgICAgID0gTm9uZQoKIyDilIDilIAgRW1iZWRkaW5nIGNhY2hlIChxdWVyeSB0ZXh0IOKGkiB2ZWN0b3IpIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAojIEVsaW1pbmF0ZXMgcmVkdW5kYW50IGZvcndhcmQgcGFzc2VzIHdoZW4gdGhlIHNhbWUgcXVlcnkgaXMgZW1iZWRkZWQgbXVsdGlwbGUKIyB0aW1lcyBhY3Jvc3MgY3ljbGVzIChBZ2VudCBBIHJlLWVtYmVkcyB0aGUgc2FtZSBjbGFpbSB0ZXh0IGVhY2ggY3ljbGUpLgpfZW1iZWRfY2FjaGU6IGRpY3QgPSB7fQoKCmRlZiBfbG9hZF9tb2RlbCgpOgogICAgIiIiTG9hZCB0b2tlbml6ZXIgKyBtb2RlbCBvbmNlLiBTdWJzZXF1ZW50IGNhbGxzIHJldHVybiBpbW1lZGlhdGVseS4iIiIKICAgIGdsb2JhbCBfdG9rZW5pemVyLCBfbW9kZWwKICAgIGlmIF90b2tlbml6ZXIgaXMgbm90IE5vbmU6CiAgICAgICAgcmV0dXJuCgogICAgcHJpbnQoZiJbRW1iZWRkZXJdIExvYWRpbmcge0VNQkVEX01PREVMfSAuLi4iKQoKICAgIGhmX3Rva2VuOiBPcHRpb25hbFtzdHJdID0gSEZfVE9LRU4gaWYgSEZfVE9LRU4gZWxzZSBOb25lCgogICAgX3Rva2VuaXplciA9IEF1dG9Ub2tlbml6ZXIuZnJvbV9wcmV0cmFpbmVkKAogICAgICAgIEVNQkVEX01PREVMLAogICAgICAgIHRydXN0X3JlbW90ZV9jb2RlPVRydWUsCiAgICAgICAgdG9rZW49aGZfdG9rZW4sCiAgICApCgogICAgX21vZGVsID0gQXV0b01vZGVsLmZyb21fcHJldHJhaW5lZCgKICAgICAgICBFTUJFRF9NT0RFTCwKICAgICAgICB0cnVzdF9yZW1vdGVfY29kZT1UcnVlLAogICAgICAgIHRva2VuPWhmX3Rva2VuLAogICAgICAgIGR0eXBlPXRvcmNoLmJmbG9hdDE2LAogICAgICAgIGRldmljZV9tYXA9ImF1dG8iLCAgICMgTVBTIG9uIEFwcGxlIFNpbGljb247IHNwaWxscyB0byBkaXNrIGlmIG1vZGVsID4gUkFNCiAgICApCiAgICBfbW9kZWwuZXZhbCgpCgogICAgcHJpbnQoZiJbRW1iZWRkZXJdIExvYWRlZCBvbiBkZXZpY2U6IHtuZXh0KF9tb2RlbC5wYXJhbWV0ZXJzKCkpLmRldmljZX0iKQoKCmRlZiBfZW5jb2RlKHRleHRzOiBMaXN0W3N0cl0pIC0+IExpc3RbTGlzdFtmbG9hdF1dOgogICAgIiIiCiAgICBDb3JlIGVuY29kZSBmdW5jdGlvbiDigJQgbWF0Y2hlcyBub3RlYm9vayBjZWxsIDE2IGV4YWN0bHk6CiAgICAgIC0gbGFzdCBoaWRkZW4gc3RhdGUgdG9rZW4gKFs6LCAtMV0pCiAgICAgIC0gTDIgbm9ybWFsaXNlZAogICAgICAtIGJmbG9hdDE2CiAgICAiIiIKICAgIF9sb2FkX21vZGVsKCkKCiAgICBpbnB1dHMgPSBfdG9rZW5pemVyKAogICAgICAgIHRleHRzLAogICAgICAgIHJldHVybl90ZW5zb3JzPSJwdCIsCiAgICAgICAgdHJ1bmNhdGlvbj1UcnVlLAogICAgICAgIHBhZGRpbmc9VHJ1ZSwKICAgICAgICBtYXhfbGVuZ3RoPUVNQkVEX01BWF9MRU5HVEgsCiAgICApLnRvKF9tb2RlbC5kZXZpY2UpCgogICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgb3V0cHV0cyA9IF9tb2RlbCgqKmlucHV0cykKCiAgICAjIExhc3QgdG9rZW4gb2YgbGFzdCBoaWRkZW4gc3RhdGUg4oCUIG1hdGNoZXMgbm90ZWJvb2sgY2VsbCAxMCBhbmQgMTYKICAgIGVtYmVkZGluZ3MgPSBvdXRwdXRzLmxhc3RfaGlkZGVuX3N0YXRlWzosIC0xXQogICAgZW1iZWRkaW5ncyA9IHRvcmNoLm5uLmZ1bmN0aW9uYWwubm9ybWFsaXplKGVtYmVkZGluZ3MsIHA9MiwgZGltPTEpCgogICAgcmV0dXJuIGVtYmVkZGluZ3MuY3B1KCkudG9saXN0KCkKCgpkZWYgZW1iZWRfcXVlcnkocXVlcnk6IHN0cikgLT4gTGlzdFtmbG9hdF06CiAgICAiIiIKICAgIEVtYmVkIGEgc2luZ2xlIHF1ZXJ5IFdJVEggdGhlIGluc3RydWN0aW9uIHByZWZpeC4KICAgIENhY2hlZCDigJQgcmVwZWF0ZWQgY2FsbHMgd2l0aCB0aGUgc2FtZSBxdWVyeSB0ZXh0IHNraXAgdGhlIGZvcndhcmQgcGFzcy4KICAgICIiIgogICAgcHJlZml4ZWQgPSBRVUVSWV9QUkVGSVggKyBxdWVyeQogICAgaWYgcHJlZml4ZWQgaW4gX2VtYmVkX2NhY2hlOgogICAgICAgIHJldHVybiBfZW1iZWRfY2FjaGVbcHJlZml4ZWRdCiAgICByZXN1bHQgPSBfZW5jb2RlKFtwcmVmaXhlZF0pWzBdCiAgICBfZW1iZWRfY2FjaGVbcHJlZml4ZWRdID0gcmVzdWx0CiAgICByZXR1cm4gcmVzdWx0CgoKZGVmIGVtYmVkX3RleHRzKHRleHRzOiBMaXN0W3N0cl0pIC0+IExpc3RbTGlzdFtmbG9hdF1dOgogICAgIiIiCiAgICBCYXRjaCBlbWJlZCBtdWx0aXBsZSB0ZXh0cyBXSVRIT1VUIHRoZSBxdWVyeSBwcmVmaXguCiAgICBVc2VkIGZvciBpbmdlc3RpbmcgbmV3IGNodW5rcyDigJQgbWF0Y2hlcyBub3RlYm9vayBpbmdlc3QgYmVoYXZpb3VyLgogICAgTm90IGNhbGxlZCBieSBNQUQgZGlyZWN0bHkgYnV0IGluY2x1ZGVkIGZvciBjb21wbGV0ZW5lc3MuCiAgICAiIiIKICAgIHJldHVybiBfZW5jb2RlKHRleHRzKQo=",
    "rag/retriever.py": "IiIiCnJhZy9yZXRyaWV2ZXIucHkg4oCUIFJlYWwgUWRyYW50IHJldHJpZXZlciBmb3IgdGhlIE1BRCBwaXBlbGluZS4KPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KCkRST1AtSU4gUkVQTEFDRU1FTlQgZm9yIG11bHRpX2FnZW50L3JhZ19zdHViLnJldHJpZXZlKCkuCgpJbnRlcmZhY2UgaXMgaWRlbnRpY2FsOgogICAgcmV0cmlldmUocXVlcnk6IHN0ciwgdG9wX2s6IGludCkgLT4gTGlzdFtFdmlkZW5jZUNodW5rXQoKVGhlIG9ubHkgZGlmZmVyZW5jZSBmcm9tIHJhZ19zdHViIGlzIHdoYXQgaGFwcGVucyBpbnNpZGUgcmV0cmlldmUoKToKICAtIHJhZ19zdHViOiBrZXl3b3JkIFRGLUlERiBzZWFyY2ggb3ZlciBhIGxvY2FsIEpTT05MIGZpbGUKICAtIHRoaXMgZmlsZTogc2VtYW50aWMgdmVjdG9yIHNlYXJjaCBhZ2FpbnN0IFFkcmFudCBjbG91ZAoKRXZlcnl0aGluZyB0aGF0IGNhbGxzIHJldHJpZXZlKCkg4oCUIGFnZW50X2EsIGFnZW50X2IsIGp1ZGdlLCBkZWJhdGVfZW5naW5lIOKAlAp3b3JrcyB3aXRob3V0IGFueSBjaGFuZ2VzLiBUaGUgaW50ZXJmYWNlIGNvbnRyYWN0IGlzIHByZXNlcnZlZCBleGFjdGx5LgoKUEFZTE9BRCBGSUVMRFMgSU4gUURSQU5UIChjb25maXJtZWQgZnJvbSBub3RlYm9vayBjZWxsIDExLzE2KToKICBjaHVua19pZCAgICDigJQgc3RyaW5nIGUuZy4gInQwX19pc29fXzI3MDAxXzIwMjJfaW5mb3NlY19fY2h1bmtfMDAwMCIKICBkb2NfaWQgICAgICDigJQgc3RyaW5nIGUuZy4gInQwX19pc29fXzI3MDAxXzIwMjJfaW5mb3NlYyIKICB0aWVyICAgICAgICDigJQgc3RyaW5nIGUuZy4gIlQwIiwgIlQxIiAgKGJhdGNoIGxhYmVsIOKAlCBOT1QgYXV0aG9yaXR5IHJhbmtpbmcpCiAgY2h1bmtfaW5kZXgg4oCUIGludAogIHRva2VuX2NvdW50IOKAlCBpbnQKICBjaHVua19tZXRob2Qg4oCUIHN0cmluZwogIHNvdXJjZV91cmwgIOKAlCBzdHJpbmcgKGRpcmVjdCBVUkwpCiAgc291cmNlICAgICAg4oCUIGRpY3QgeyJkb2N1bWVudF9uYW1lIjogIi4uLiIsICJwZGZfdXJsIjogIi4uLiJ9CiAgdGV4dCAgICAgICAg4oCUIHN0cmluZyAoY2h1bmsgY29udGVudCkKCkhPVyBUTyBBQ1RJVkFURToKICBJbiBtdWx0aV9hZ2VudC9yYWdfc3R1Yi5weSwgcmVwbGFjZSB0aGUgcmV0cmlldmUoKSBmdW5jdGlvbiBib2R5IHdpdGg6CiAgICBmcm9tIHJhZy5yZXRyaWV2ZXIgaW1wb3J0IHJldHJpZXZlCiAgU2VlIHRoZSBvbmUtbGluZSBzd2FwIGluc3RydWN0aW9ucyBhdCB0aGUgYm90dG9tIG9mIHRoaXMgZmlsZS4KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gdHlwaW5nIGltcG9ydCBMaXN0Cgpmcm9tIHFkcmFudF9jbGllbnQgaW1wb3J0IFFkcmFudENsaWVudAoKZnJvbSByYWcuY29uZmlnIGltcG9ydCAoCiAgICBRRFJBTlRfVVJMLCBRRFJBTlRfQVBJX0tFWSwgUURSQU5UX1RJTUVPVVQsCiAgICBDT0xMRUNUSU9OX05BTUUsCikKZnJvbSByYWcuZW1iZWRkZXIgaW1wb3J0IGVtYmVkX3F1ZXJ5CgojIEltcG9ydCBFdmlkZW5jZUNodW5rIGZyb20gbXVsdGlfYWdlbnQg4oCUIHNhbWUgc2NoZW1hLCBubyBkdXBsaWNhdGlvbgpmcm9tIG11bHRpX2FnZW50Lm1vZGVscyBpbXBvcnQgRXZpZGVuY2VDaHVuawoKIyDilIDilIAgU2luZ2xldG9uIFFkcmFudCBjbGllbnQg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACl9jbGllbnQ6IFFkcmFudENsaWVudCA9IE5vbmUKCgpkZWYgX2dldF9jbGllbnQoKSAtPiBRZHJhbnRDbGllbnQ6CiAgICAiIiJSZXR1cm4gdGhlIFFkcmFudCBjbGllbnQsIGNyZWF0aW5nIGl0IG9uIGZpcnN0IGNhbGwuIiIiCiAgICBnbG9iYWwgX2NsaWVudAogICAgaWYgX2NsaWVudCBpcyBOb25lOgogICAgICAgIHByaW50KGYiW1JBR10gQ29ubmVjdGluZyB0byBRZHJhbnQgYXQge1FEUkFOVF9VUkx9IC4uLiIpCiAgICAgICAgX2NsaWVudCA9IFFkcmFudENsaWVudCgKICAgICAgICAgICAgdXJsPVFEUkFOVF9VUkwsCiAgICAgICAgICAgIGFwaV9rZXk9UURSQU5UX0FQSV9LRVksCiAgICAgICAgICAgIHRpbWVvdXQ9UURSQU5UX1RJTUVPVVQsCiAgICAgICAgKQogICAgICAgIHByaW50KGYiW1JBR10gQ29ubmVjdGVkLiBDb2xsZWN0aW9uOiB7Q09MTEVDVElPTl9OQU1FfSIpCiAgICByZXR1cm4gX2NsaWVudAoKCmRlZiByZXRyaWV2ZShxdWVyeTogc3RyLCB0b3BfazogaW50ID0gNSkgLT4gTGlzdFtFdmlkZW5jZUNodW5rXToKICAgICIiIgogICAgUmV0cmlldmUgdG9wX2sgcmVndWxhdG9yeSBjaHVua3MgZnJvbSBRZHJhbnQgZm9yIHRoZSBnaXZlbiBxdWVyeS4KCiAgICBTdGVwczoKICAgICAgMS4gRW1iZWQgdGhlIHF1ZXJ5IHVzaW5nIE5lbW90cm9uLThCIFdJVEggdGhlIGluc3RydWN0aW9uIHByZWZpeAogICAgICAyLiBSdW4gY29zaW5lIHNpbWlsYXJpdHkgc2VhcmNoIGFnYWluc3QgdGhlIFFkcmFudCBjb2xsZWN0aW9uCiAgICAgIDMuIE1hcCByZXN1bHRzIHRvIEV2aWRlbmNlQ2h1bmsgb2JqZWN0cwoKICAgIFRoaXMgaXMgYSBkcm9wLWluIHJlcGxhY2VtZW50IGZvciByYWdfc3R1Yi5yZXRyaWV2ZSgpLgogICAgVGhlIHNpZ25hdHVyZSBhbmQgcmV0dXJuIHR5cGUgYXJlIGlkZW50aWNhbC4KCiAgICBBcmdzOgogICAgICAgIHF1ZXJ5ICDigJQgdGhlIHRleHQgdG8gc2VhcmNoIGZvciAoY2xhaW0gdGV4dCBvciB1c2VyIHF1ZXJ5KQogICAgICAgIHRvcF9rICDigJQgbnVtYmVyIG9mIGNodW5rcyB0byByZXR1cm4KCiAgICBSZXR1cm5zOgogICAgICAgIExpc3RbRXZpZGVuY2VDaHVua10gb3JkZXJlZCBieSByZWxldmFuY2UgKG1vc3QgcmVsZXZhbnQgZmlyc3QpCiAgICAiIiIKICAgICMgU3RlcCAxOiBlbWJlZCBxdWVyeSB3aXRoIHByZWZpeAogICAgcXVlcnlfdmVjdG9yID0gZW1iZWRfcXVlcnkocXVlcnkpCgogICAgIyBTdGVwIDI6IHNlYXJjaCBRZHJhbnQKICAgIGNsaWVudCA9IF9nZXRfY2xpZW50KCkKICAgIHJlc3VsdHMgPSBjbGllbnQucXVlcnlfcG9pbnRzKAogICAgICAgIGNvbGxlY3Rpb25fbmFtZT1DT0xMRUNUSU9OX05BTUUsCiAgICAgICAgcXVlcnk9cXVlcnlfdmVjdG9yLAogICAgICAgIGxpbWl0PXRvcF9rLAogICAgICAgIHdpdGhfcGF5bG9hZD1UcnVlLAogICAgICAgIHdpdGhfdmVjdG9ycz1GYWxzZSwKICAgICkucG9pbnRzCgogICAgIyBTdGVwIDM6IG1hcCB0byBFdmlkZW5jZUNodW5rCiAgICBjaHVua3M6IExpc3RbRXZpZGVuY2VDaHVua10gPSBbXQogICAgZm9yIHIgaW4gcmVzdWx0czoKICAgICAgICBwID0gci5wYXlsb2FkCgogICAgICAgICMgc291cmNlIGZpZWxkIGlzIGEgZGljdCBpbiB0aGlzIGNvcnB1czogeyJkb2N1bWVudF9uYW1lIjogLi4uLCAicGRmX3VybCI6IC4uLn0KICAgICAgICAjIFdlIHN0b3JlIGRvY3VtZW50X25hbWUgYXMgdGhlIHNvdXJjZSBzdHJpbmcgZm9yIGRpc3BsYXkvY2l0YXRpb24gcHVycG9zZXMuCiAgICAgICAgIyBGYWxscyBiYWNrIHRvIHNvdXJjZV91cmwgb3IgZG9jX2lkIGlmIHNvdXJjZSBkaWN0IGlzIG5vdCBwcmVzZW50LgogICAgICAgIHNvdXJjZV9vYmogPSBwLmdldCgic291cmNlIiwge30pCiAgICAgICAgaWYgaXNpbnN0YW5jZShzb3VyY2Vfb2JqLCBkaWN0KToKICAgICAgICAgICAgc291cmNlX3N0ciA9IHNvdXJjZV9vYmouZ2V0KCJkb2N1bWVudF9uYW1lIikgb3IgcC5nZXQoImRvY19pZCIsICJ1bmtub3duIikKICAgICAgICBlbHNlOgogICAgICAgICAgICBzb3VyY2Vfc3RyID0gc3RyKHNvdXJjZV9vYmopIGlmIHNvdXJjZV9vYmogZWxzZSBwLmdldCgiZG9jX2lkIiwgInVua25vd24iKQoKICAgICAgICBjaHVua3MuYXBwZW5kKEV2aWRlbmNlQ2h1bmsoCiAgICAgICAgICAgIGNodW5rX2lkPXN0cihwLmdldCgiY2h1bmtfaWQiLCByLmlkKSksCiAgICAgICAgICAgIHRleHQ9cC5nZXQoInRleHQiLCAiIiksCiAgICAgICAgICAgIHNvdXJjZT1zb3VyY2Vfc3RyLAogICAgICAgICAgICB0aWVyPTEsICAgIyB0aWVyIGZpZWxkICgiVDAiLyJUMSIpIGlzIGEgYmF0Y2ggbGFiZWwsIG5vdCBhdXRob3JpdHkgcmFua2luZwogICAgICAgICAgICAgICAgICAgICAgIyBzdG9yZWQgYXMgMSAobmV1dHJhbCkg4oCUIG5vdCB1c2VkIGZvciBmaWx0ZXJpbmcgaW4gTUFEIHYwLjEKICAgICAgICApKQoKICAgIHJldHVybiBjaHVua3MKCgpkZWYgaGVhbHRoX2NoZWNrKCkgLT4gZGljdDoKICAgICIiIgogICAgVmVyaWZ5IFFkcmFudCBjb25uZWN0aW9uIGFuZCBjb2xsZWN0aW9uIGFyZSByZWFjaGFibGUuCiAgICBDYWxsIHRoaXMgYXQgc3RhcnR1cCBiZWZvcmUgcnVubmluZyBNQUQgdG8gY2F0Y2ggY29uZmlnIGVycm9ycyBlYXJseS4KCiAgICBSZXR1cm5zIGRpY3Qgd2l0aCBzdGF0dXMgYW5kIGNvbGxlY3Rpb24gaW5mby4KICAgICIiIgogICAgdHJ5OgogICAgICAgIGNsaWVudCA9IF9nZXRfY2xpZW50KCkKICAgICAgICBpbmZvID0gY2xpZW50LmdldF9jb2xsZWN0aW9uKENPTExFQ1RJT05fTkFNRSkKICAgICAgICByZXR1cm4gewogICAgICAgICAgICAic3RhdHVzIjogIm9rIiwKICAgICAgICAgICAgImNvbGxlY3Rpb24iOiBDT0xMRUNUSU9OX05BTUUsCiAgICAgICAgICAgICJwb2ludHNfY291bnQiOiBpbmZvLnBvaW50c19jb3VudCwKICAgICAgICB9CiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgcmV0dXJuIHsKICAgICAgICAgICAgInN0YXR1cyI6ICJlcnJvciIsCiAgICAgICAgICAgICJlcnJvciI6IHN0cihlKSwKICAgICAgICB9CgoKIyDilIDilIAgSE9XIFRPIEFDVElWQVRFIFRISVMgSU4gTUFEIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAojCiMgSW4gbXVsdGlfYWdlbnQvcmFnX3N0dWIucHksIHJlcGxhY2UgdGhlIGVudGlyZSByZXRyaWV2ZSgpIGZ1bmN0aW9uIHdpdGg6CiMKIyAgICAgZnJvbSByYWcucmV0cmlldmVyIGltcG9ydCByZXRyaWV2ZSAgIyBub3FhOiBGNDAxCiMKIyBUaGF0IHNpbmdsZSBpbXBvcnQgbWFrZXMgcmFnX3N0dWIucmV0cmlldmUgcG9pbnQgdG8gdGhpcyByZWFsIGltcGxlbWVudGF0aW9uLgojIGFnZW50X2EsIGFnZW50X2IsIGp1ZGdlLCBkZWJhdGVfZW5naW5lIGNhbGwgcmFnX3N0dWIucmV0cmlldmUoKSB1bmNoYW5nZWQuCiMgTm90aGluZyBlbHNlIG5lZWRzIHRvIGNoYW5nZSBhbnl3aGVyZSBpbiB0aGUgTUFEIGNvZGViYXNlLgo=",
    "multi_agent/__init__.py": "IyBtdWx0aV9hZ2VudCDigJQgTUFEIChNdWx0aS1BZ2VudCBEZWJhdGUpIHZlcmlmaWNhdGlvbiBwaXBlbGluZQojIEd1YXJkcmFpbHMgR2F0ZXdheSDCtyBTSlNVIENTMjk4QiDCtyAyMDI1LTI2Cg==",
    "multi_agent/models.py": "IiIiCm1vZGVscy5weSDigJQgUHlkYW50aWMgc2NoZW1hcyBmb3IgdGhlIE1BRCBwaXBlbGluZS4KVGhlc2UgYXJlIHRoZSBjYW5vbmljYWwgZGF0YSBjb250cmFjdHMgYmV0d2VlbiBldmVyeSBtb2R1bGUuCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIGVudW0gaW1wb3J0IEVudW0KZnJvbSB0eXBpbmcgaW1wb3J0IExpc3QsIE9wdGlvbmFsCmZyb20gcHlkYW50aWMgaW1wb3J0IEJhc2VNb2RlbCwgRmllbGQKCgojIOKUgOKUgCBFbnVtcyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKCmNsYXNzIFZlcmRpY3Qoc3RyLCBFbnVtKToKICAgIFNVUFBPUlRFRCAgICAgPSAiU1VQUE9SVEVEIgogICAgUEFSVElBTCAgICAgICA9ICJQQVJUSUFMIgogICAgTk9UX1NVUFBPUlRFRCA9ICJOT1RfU1VQUE9SVEVEIgogICAgSURLICAgICAgICAgICA9ICJJREsiCgoKY2xhc3MgQ2hhbGxlbmdlVHlwZShzdHIsIEVudW0pOgogICAgQ0hVTktfQ1VSUkVOQ1kgICAgID0gIkNIVU5LX0NVUlJFTkNZIiAgICAgICAjIElzIHRoZSBjaXRlZCBjaHVuayBjdXJyZW50IC8gbm90IHN1cGVyc2VkZWQ/CiAgICBKVVJJU0RJQ1RJT05fU0NPUEUgPSAiSlVSSVNESUNUSU9OX1NDT1BFIiAgICMgRG9lcyB0aGlzIHJlZ3VsYXRpb24gYXBwbHkgdG8gdGhpcyBqdXJpc2RpY3Rpb24/CiAgICBFWENFUFRJT05fRVhJU1RFTkNFPSAiRVhDRVBUSU9OX0VYSVNURU5DRSIgICMgRG9lcyBhbiBleGNlcHRpb24gLyBjYXJ2ZS1vdXQgYXBwbHk/CiAgICBHQVBfRklORElORyAgICAgICAgPSAiR0FQX0ZJTkRJTkciICAgICAgICAgICMgTWlzc2luZyByZWd1bGF0b3J5IGNvbnRleHQgQWdlbnQgQSBvdmVybG9va2VkCgoKY2xhc3MgUm91dGluZ0RlY2lzaW9uKHN0ciwgRW51bSk6CiAgICBERUxJVkVSICAgICAgPSAiREVMSVZFUiIgICAgICAgIyBhZ2dyZWdhdGUgPiAwLjgsIGFsbCBtYXRlcmlhbCB2PTEuMAogICAgUkVUUlkgICAgICAgID0gIlJFVFJZIiAgICAgICAgICMgYWdncmVnYXRlIDAuNOKAkzAuOCDihpIgc2VuZCBjb3JyZWN0aW9uIHRvIExMTQogICAgSEFSRF9CTE9DSyAgID0gIkhBUkRfQkxPQ0siICAgICMgYW55IGlzX21hdGVyaWFsIGNsYWltIHdpdGgganVkZ2Ugc2NvcmU9MC4wCiAgICBIVU1BTl9SRVZJRVcgPSAiSFVNQU5fUkVWSUVXIiAgIyBhZ2dyZWdhdGUgPCAwLjQKCgojIOKUgOKUgCBDb3JlIHNjaGVtYXMg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACgpjbGFzcyBDbGFpbShCYXNlTW9kZWwpOgogICAgIiIiQXRvbWljIHZlcmlmaWFibGUgc3RhdGVtZW50IGV4dHJhY3RlZCBmcm9tIHRoZSBMTE0gYW5zd2VyLiIiIgogICAgY2xhaW1faWQ6ICAgICAgIGludAogICAgY2xhaW1fdGV4dDogICAgIHN0cgogICAgaXNfbWF0ZXJpYWw6ICAgIGJvb2wgPSBUcnVlICAgICAgICAgICMgVHJ1ZSA9IHJlZ3VsYXRvcnkgb2JsaWdhdGlvbiAvIHBlbmFsdHkgLyB0aHJlc2hvbGQKICAgIGNvbmZpZGVuY2U6ICAgICBmbG9hdCA9IEZpZWxkKGRlZmF1bHQ9MC43LCBnZT0wLjAsIGxlPTEuMCkgICMgQWdlbnQgQSdzIGNhbGlicmF0ZWQgYmVsaWVmCiAgICB2ZXJkaWN0OiAgICAgICAgT3B0aW9uYWxbVmVyZGljdF0gPSBOb25lCiAgICBldmlkZW5jZV9jaHVua3M6IExpc3Rbc3RyXSA9IFtdICAgICAjIGNodW5rX2lkcyB1c2VkIGFzIGV2aWRlbmNlCiAgICByZWFzb25pbmc6ICAgICAgc3RyID0gIiIKCgpjbGFzcyBFdmlkZW5jZUNodW5rKEJhc2VNb2RlbCk6CiAgICAiIiJBIHNpbmdsZSByZWd1bGF0b3J5IGRvY3VtZW50IGNodW5rIGZyb20gdGhlIFJBRyBwaXBlbGluZS4iIiIKICAgIGNodW5rX2lkOiBzdHIKICAgIHRleHQ6ICAgICBzdHIKICAgIHNvdXJjZTogICBzdHIKICAgIHRpZXI6ICAgICBpbnQgPSAxICAgIyAxPXByaW1hcnkgbGF3LCAyPWd1aWRhbmNlLCAzPWZyYW1ld29yaywgND1jb21tZW50YXJ5CgoKY2xhc3MgQ2hhbGxlbmdlKEJhc2VNb2RlbCk6CiAgICAiIiJBIHNpbmdsZSBjaGFsbGVuZ2UgcmFpc2VkIGJ5IEFnZW50IEIgYWdhaW5zdCBhIHNwZWNpZmljIGNsYWltLiIiIgogICAgY2xhaW1faWQ6ICAgICAgICBpbnQKICAgIGNoYWxsZW5nZV90eXBlOiAgQ2hhbGxlbmdlVHlwZQogICAgY2hhbGxlbmdlX3RleHQ6ICBzdHIKICAgIGV2aWRlbmNlX2NodW5rczogTGlzdFtzdHJdID0gW10gICAgICAgICAgIyBjaHVua19pZHMgQiByZXRyaWV2ZWQgZm9yIHRoaXMgY2hhbGxlbmdlCiAgICBzdWdnZXN0ZWRfdmVyZGljdDogT3B0aW9uYWxbVmVyZGljdF0gPSBOb25lCgoKY2xhc3MgRGViYXRlQ3ljbGUoQmFzZU1vZGVsKToKICAgICIiIkZ1bGwgcmVjb3JkIG9mIG9uZSBjaGFsbGVuZ2UtcmV2aXNpb24gY3ljbGUuIiIiCiAgICBjeWNsZV9udW1iZXI6ICAgICAgIGludAogICAgYWdlbnRfYV9yZXBvcnQ6ICAgICBMaXN0W0NsYWltXSAgICAgICAjIEEncyB2ZXJkaWN0cyBhdCBzdGFydCBvZiBjeWNsZQogICAgYWdlbnRfYl9jaGFsbGVuZ2VzOiBMaXN0W0NoYWxsZW5nZV0gICAjIEIncyBjaGFsbGVuZ2VzCiAgICBhZ2VudF9hX3JldmlzZWQ6ICAgIExpc3RbQ2xhaW1dICAgICAgICMgQSdzIHVwZGF0ZWQgdmVyZGljdHMgYWZ0ZXIgQidzIGNoYWxsZW5nZXMKICAgIGNvbmZpZGVuY2Vfc2lnbmFsOiAgZmxvYXQgICAgICAgICAgICAgIyBtaW4gYWdncmVnYXRpb24gb2YgbWF0ZXJpYWwgY2xhaW0gY29uZmlkZW5jZXMKCgpjbGFzcyBKdWRnZVZlcmRpY3QoQmFzZU1vZGVsKToKICAgICIiIkp1ZGdlJ3MgaW5kZXBlbmRlbnQgc2NvcmUgZm9yIGEgc2luZ2xlIGNsYWltLiIiIgogICAgY2xhaW1faWQ6ICAgaW50CiAgICBjbGFpbV90ZXh0OiBzdHIKICAgIGlzX21hdGVyaWFsOiBib29sCiAgICBzY29yZTogICAgICBmbG9hdCAgICMgMS4wID0gZnVsbHkgc3VwcG9ydGVkLCAwLjUgPSBwYXJ0aWFsLCAwLjAgPSB1bnN1cHBvcnRlZAogICAgcmVhc29uaW5nOiAgc3RyCgoKY2xhc3MgTUFET3V0cHV0KEJhc2VNb2RlbCk6CiAgICAiIiJGdWxsIG91dHB1dCBvZiB0aGUgTUFEIHBpcGVsaW5lIOKAlCByZXR1cm5lZCB0byBjYWxsZXIgLyBBUEkuIiIiCiAgICBxdWVyeTogICAgICAgICAgICAgICBzdHIKICAgIGxsbV9hbnN3ZXI6ICAgICAgICAgIHN0cgogICAgY2xhaW1zOiAgICAgICAgICAgICAgTGlzdFtDbGFpbV0KICAgIGRlYmF0ZV9jeWNsZXM6ICAgICAgIExpc3RbRGViYXRlQ3ljbGVdCiAgICBldmlkZW5jZV9wb29sOiAgICAgICBMaXN0W0V2aWRlbmNlQ2h1bmtdCiAgICBqdWRnZV92ZXJkaWN0czogICAgICBMaXN0W0p1ZGdlVmVyZGljdF0KICAgIGNvcnJlY3Rpb25fc2lnbmFsOiAgIE9wdGlvbmFsW3N0cl0gICAgICAgIyBTZW50IHRvIExMTSBvbiByZXRyeQogICAgcm91dGluZ19kZWNpc2lvbjogICAgc3RyICAgICAgICAgICAgICAgICAjIFJvdXRpbmdEZWNpc2lvbiB2YWx1ZQogICAgYWdncmVnYXRlX2NvbmZpZGVuY2U6IGZsb2F0CiAgICBkZWJhdGVfdHJhbnNjcmlwdDogICBzdHIgICAgICAgICAgICAgICAgICMgSHVtYW4tcmVhZGFibGUgZnVsbCB0cmFuc2NyaXB0CiAgICAjIFN0b3JhZ2UgSURzIOKAlCBuZWVkZWQgZm9yIGZlZWRiYWNrIGxvb3AgdG8gam9pbiB0YWJsZXMKICAgIHF1ZXJ5X2lkOiAgICAgICAgICAgIHN0ciA9ICIiICAgICAgICAgICAgIyBVVUlEIOKAlCBqb2lucyBhbGwgNCB0YWJsZXMKICAgIHJvbGxvdXRfaWQ6ICAgICAgICAgIHN0ciA9ICIiICAgICAgICAgICAgIyBVVUlEIOKAlCBmb3IgR1JQTyBtdWx0aS1yb2xsb3V0IGNvbXBhcmlzb24K",
    "multi_agent/storage.py": "...",
    "multi_agent/rag_stub.py": "...",
    "multi_agent/debate_engine.py": "...",
    "multi_agent/mad_pipeline.py": "IiIiCm1hZF9waXBlbGluZS5weSDigJQgVG9wLWxldmVsIE1BRCBwaXBlbGluZSBvcmNoZXN0cmF0b3IKPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CgpFTlRSWSBQT0lOVAotLS0tLS0tLS0tLQpydW5fbWFkKHF1ZXJ5LCBsbG1fYW5zd2VyKSDihpIgTUFET3V0cHV0CgpUaGlzIGlzIHRoZSBzaW5nbGUgZnVuY3Rpb24gdGhlIGdhdGV3YXkgKG9yIEFQSSkgY2FsbHMuIEl0IG9yY2hlc3RyYXRlcwp0aGUgZnVsbCBwaXBlbGluZSBhbmQgaGFuZGxlcyBzdG9yYWdlIGF0IHRoZSBzdGFydCBhbmQgZW5kLgoKQ09NUExFVEUgRkxPVwotLS0tLS0tLS0tLS0tCjEuICBHZW5lcmF0ZSByb2xsb3V0X2lkIChVVUlEKSDigJQgbmVlZGVkIGZvciBHUlBPIG11bHRpLXJvbGxvdXQgY29tcGFyaXNvbgoyLiAgRXh0cmFjdCBhdG9taWMgY2xhaW1zIGZyb20gTExNIGFuc3dlcgozLiAgW1NUT1JBR0VdIFdyaXRlIHF1ZXJ5IHJvdyAoVEFCTEUgMSkKNC4gIFJ1biBkZWJhdGUgZW5naW5lICh3aGljaCB3cml0ZXMgY2xhaW1zICsgYXR0YWNrcyByb3dzIGludGVybmFsbHkpCjUuICBSdW4gSnVkZ2UgZXZhbHVhdGlvbiAocGFydGlhbGx5IGJsaW5kIOKAlCBjb25maWRlbmNlIHNjb3JlcyBzdHJpcHBlZCkKNi4gIFtTVE9SQUdFXSBXcml0ZSBqdWRnZV92ZXJkaWN0cyAoVEFCTEUgNCkKNy4gIENvbXB1dGUgcm91dGluZyBkZWNpc2lvbiAoaGFyZCBydWxlIGZpcnN0LCB0aGVuIGFnZ3JlZ2F0ZSBzY29yZSkKOC4gIFtTVE9SQUdFXSBVcGRhdGUgcXVlcnkgd2l0aCBmaW5hbCBDU0Ugc2NvcmUgKyByb3V0aW5nIGRlY2lzaW9uCjkuICBCdWlsZCB0cmFuc2NyaXB0CjEwLiBSZXR1cm4gTUFET3V0cHV0CgpST1VUSU5HIExPR0lDCi0tLS0tLS0tLS0tLS0KSGFyZCBydWxlIChjaGVja2VkIEZJUlNULCBvdmVycmlkZXMgZXZlcnl0aGluZyk6CiAgQW55IGlzX21hdGVyaWFsIGNsYWltIHdpdGgganVkZ2Ugc2NvcmUgPSAwLjAg4oaSIEhBUkRfQkxPQ0sg4oaSIGh1bWFuIHJldmlldwogIEEgZmFicmljYXRlZCByZWd1bGF0b3J5IHN0YW5kYXJkIChlLmcuICJISVBBQSBtYW5kYXRlcyBBRVMtMjU2IikgdGhhdAogIHNjb3JlcyAwLjAgY2Fubm90IHJlYWNoIHRoZSB1c2VyIHZpYSBhbnkgcmV0cnkgcGF0aC4KClNvZnQgcnVsZXMgKGFmdGVyIGhhcmQgcnVsZSBwYXNzZXMpOgogIGFnZ3JlZ2F0ZSA+IDAuOCDihpIgREVMSVZFUgogIGFnZ3JlZ2F0ZSAwLjTigJMwLjgg4oaSIFJFVFJZIChzZW5kIGNvcnJlY3Rpb24gc2lnbmFsIHRvIExMTSkKICBhZ2dyZWdhdGUgPCAwLjQg4oaSIEhVTUFOX1JFVklFVwoKV0hBVCBJUyBBR0dSRUdBVEUgQ09ORklERU5DRT8KLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCkN1cnJlbnRseTogbWluLWFnZ3JlZ2F0ZWQganVkZ2Ugc2NvcmVzIGZvciBtYXRlcmlhbCBjbGFpbXMgKDcwJSB3ZWlnaHQpCiAgICAgICAgICAgKyBtZWFuIGp1ZGdlIHNjb3JlcyBmb3Igbm9uLW1hdGVyaWFsIGNsYWltcyAoMzAlIHdlaWdodCkKCkZ1bGwgQ1NFIGZvcm11bGEgKFBoYXNlIDIg4oCUIHdoZW4gRGVlcEV2YWwgTGF5ZXIgMSBpcyBpbnRlZ3JhdGVkKToKICBmaW5hbCA9IDAuMzAqRl9sbG0gKyAwLjI1KigxLUhfbGxtKSArIDAuMTAqcmVsZXZhbmN5ICsgMC4zNSpqdWRnZV9ldmFsX3Njb3JlCgpUaGUgcGFwZXIgbXVzdCByZXBvcnQgdGhpcyBhcyAianVkZ2Utb25seSBhZ2dyZWdhdGUgKHYwLjEpIiBub3QgdGhlIGZ1bGwgZm9ybXVsYS4KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCB1dWlkCmZyb20gdHlwaW5nIGltcG9ydCBMaXN0LCBPcHRpb25hbCwgVHVwbGUKCmZyb20gbXVsdGlfYWdlbnQgaW1wb3J0IHN0b3JhZ2UKZnJvbSBtdWx0aV9hZ2VudC5jbGFpbV9leHRyYWN0b3IgaW1wb3J0IGV4dHJhY3RfY2xhaW1zCmZyb20gbXVsdGlfYWdlbnQuY29uZmlnIGltcG9ydCAoCiAgICBNQVhfQ1lDTEVTLAogICAgQ09ORklERU5DRV9USFJFU0hPTERfSElHSCwKICAgIENPTkZJREVOQ0VfVEhSRVNIT0xEX0xPVywKKQpmcm9tIG11bHRpX2FnZW50LmRlYmF0ZV9lbmdpbmUgaW1wb3J0IHJ1bl9kZWJhdGUsIGJ1aWxkX3RyYW5zY3JpcHQKZnJvbSBtdWx0aV9hZ2VudC5qdWRnZSBpbXBvcnQganVkZ2VfY2xhaW1zCmZyb20gbXVsdGlfYWdlbnQubW9kZWxzIGltcG9ydCBDbGFpbSwgSnVkZ2VWZXJkaWN0LCBNQURPdXRwdXQKCgpkZWYgcnVuX21hZChxdWVyeTogc3RyLCBsbG1fYW5zd2VyOiBzdHIpIC0+IE1BRE91dHB1dDoKICAgICIiIgogICAgRnVsbCBNQUQgcGlwZWxpbmUgd2l0aCBjb21wbGV0ZSBzdG9yYWdlIGludGVncmF0aW9uLgoKICAgIEV2ZXJ5IGludGVybWVkaWF0ZSByZXN1bHQgaXMgd3JpdHRlbiB0byBTUUxpdGUgc28gdGhlIEdSUE8KICAgIGZlZWRiYWNrIGxvb3AgY2FuIGNvbnN1bWUgaXQgYWZ0ZXIgdGhlIHJ1biBjb21wbGV0ZXMuCiAgICAiIiIKICAgICMg4pSA4pSAIDEuIEdFTkVSQVRFIFJPTExPVVRfSUQg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACiAgICAjIHJvbGxvdXRfaWQgaXMgYSBVVUlEIHRoYXQgdW5pcXVlbHkgaWRlbnRpZmllcyBUSElTIHJ1biBvZiB0aGUgcGlwZWxpbmUKICAgICMgZm9yIHRoaXMgcXVlcnkuIEZvciBHUlBPIHRyYWluaW5nLCB5b3UgcnVuIHRoZSBzYW1lIHF1ZXJ5IG11bHRpcGxlIHRpbWVzCiAgICAjIChtdWx0aXBsZSByb2xsb3V0cykgYW5kIGNvbXBhcmUgQnJpZXIgcmV3YXJkcyBhY3Jvc3Mgcm9sbG91dHMgdG8gY29tcHV0ZQogICAgIyBncnBvX2FkdmFudGFnZSA9IHJvbGxvdXRfdG90YWwgLSBtZWFuX2Fjcm9zc19yb2xsb3V0cy4KICAgIHJvbGxvdXRfaWQgPSBzdHIodXVpZC51dWlkNCgpKQogICAgcXVlcnlfaWQgICA9IHN0cih1dWlkLnV1aWQ0KCkpCgogICAgX2Jhbm5lcigiTUFEIFBJUEVMSU5FIikKICAgIHByaW50KGYiICBxdWVyeV9pZCAgOiB7cXVlcnlfaWR9IikKICAgIHByaW50KGYiICByb2xsb3V0X2lkOiB7cm9sbG91dF9pZH0iKQogICAgcHJpbnQoZiIgIFF1ZXJ5ICAgICA6IHtxdWVyeX0iKQogICAgcHJpbnQoZiIgIEFuc3dlciAgICA6IHtsbG1fYW5zd2VyWzoxMjBdfXsnLi4uJyBpZiBsZW4obGxtX2Fuc3dlcikgPiAxMjAgZWxzZSAnJ30iKQoKICAgICMg4pSA4pSAIDIuIElOSVRJQUxJU0UgU1RPUkFHRSDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKICAgIHN0b3JhZ2UuaW5pdF9kYigpCgogICAgIyDilIDilIAgMy4gRVhUUkFDVCBDTEFJTVMg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACiAgICBwcmludCgiXG5bU3RlcCAxLzRdIEV4dHJhY3RpbmcgYXRvbWljIGNsYWltcy4uLiIpCiAgICBjbGFpbXMgID0gZXh0cmFjdF9jbGFpbXMocXVlcnksIGxsbV9hbnN3ZXIpCiAgICBuX21hdCAgID0gc3VtKDEgZm9yIGMgaW4gY2xhaW1zIGlmIGMuaXNfbWF0ZXJpYWwpCiAgICBwcmludChmIiAgICAgICAgICAge2xlbihjbGFpbXMpfSBjbGFpbXMgKHtuX21hdH0gbWF0ZXJpYWwsICIKICAgICAgICAgIGYie2xlbihjbGFpbXMpLW5fbWF0fSBjb250ZXh0dWFsKSIpCgogICAgIyDilIDilIAgNC4gV1JJVEUgUVVFUlkgUk9XIChUQUJMRSAxKSDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKICAgICMgV3JpdHRlbiBCRUZPUkUgZGViYXRlIHN0YXJ0cy4gZmluYWxfY3NlX3Njb3JlICsgcm91dGluZ19kZWNpc2lvbiA9IE5VTEwuCiAgICAjIFVwZGF0ZWQgYXQgZW5kIG9mIHBpcGVsaW5lIHdpdGggYWN0dWFsIHZhbHVlcy4KICAgIHN0b3JhZ2Uud3JpdGVfcXVlcnkoCiAgICAgICAgcXVlcnlfaWQ9cXVlcnlfaWQsCiAgICAgICAgcm9sbG91dF9pZD1yb2xsb3V0X2lkLAogICAgICAgIHF1ZXJ5X3RleHQ9cXVlcnksCiAgICAgICAgbGxtX2Fuc3dlcj1sbG1fYW5zd2VyLAogICAgICAgIGNodW5rX2lkcz1bXSwgICMgcG9wdWxhdGVkIGFmdGVyIGRlYmF0ZSBvbmNlIGV2aWRlbmNlX3Bvb2wgaXMga25vd24KICAgICkKCiAgICAjIOKUgOKUgCA1LiBSVU4gREVCQVRFIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAogICAgIyBkZWJhdGVfZW5naW5lIGhhbmRsZXMgYWxsIGludGVybmFsIHN0b3JhZ2Ugd3JpdGVzOgogICAgIyAgIC0gY2xhaW1zIHRhYmxlOiBwb3N0X3N0ZXBfQSwgcG9zdF9jeWNsZTEsIHBvc3RfY3ljbGUyCiAgICAjICAgLSBhdHRhY2tzIHRhYmxlOiBwX2JlZm9yZSBhdCBjaGFsbGVuZ2UgdGltZSwgcF9hZnRlciBhZnRlciByZXZpc2lvbgogICAgcHJpbnQoZiJcbltTdGVwIDIvNF0gUnVubmluZyB7TUFYX0NZQ0xFU30tY3ljbGUgZGViYXRlLi4uIikKICAgIGN5Y2xlcywgZmluYWxfY2xhaW1zLCBldmlkZW5jZV9wb29sID0gcnVuX2RlYmF0ZSgKICAgICAgICBxdWVyeT1xdWVyeSwKICAgICAgICBjbGFpbXM9Y2xhaW1zLAogICAgICAgIHF1ZXJ5X2lkPXF1ZXJ5X2lkLAogICAgICAgIHJvbGxvdXRfaWQ9cm9sbG91dF9pZCwKICAgICAgICBtYXhfY3ljbGVzPU1BWF9DWUNMRVMsCiAgICApCgogICAgIyDilIDilIAgNi4gSlVER0UgRVZBTFVBVElPTiDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKICAgICMgSnVkZ2UgaXMgUEFSVElBTExZIEJMSU5EOgogICAgIyAgIFNFRVM6ICAgICBmaW5hbCB2ZXJkaWN0cyArIGV2aWRlbmNlIHBvb2wgKyB1c2VyIHF1ZXJ5CiAgICAjICAgRE9FUyBOT1QgU0VFOiBjb25maWRlbmNlIHNjb3JlcyAoc3RyaXBwZWQgdG8gcHJldmVudCBhbmNob3JpbmcpCiAgICAjICAgICAgICAgICAgICAgICBBZ2VudCBCJ3MgY2hhbGxlbmdlIGZyYW1pbmcKICAgIHByaW50KCJcbltTdGVwIDMvNF0gSnVkZ2UgZXZhbHVhdGlvbiAocGFydGlhbGx5IGJsaW5kKS4uLiIpCiAgICBqdWRnZV92ZXJkaWN0cywgY29ycmVjdGlvbl9zaWduYWwgPSBqdWRnZV9jbGFpbXMoCiAgICAgICAgcXVlcnk9cXVlcnksCiAgICAgICAgZmluYWxfY2xhaW1zPWZpbmFsX2NsYWltcywKICAgICAgICBldmlkZW5jZV9wb29sPWV2aWRlbmNlX3Bvb2wsCiAgICApCiAgICBfcHJpbnRfanVkZ2VfdmVyZGljdHMoanVkZ2VfdmVyZGljdHMpCgogICAgIyDilIDilIAgNy4gV1JJVEUgSlVER0UgVkVSRElDVFMgKFRBQkxFIDQpIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAogICAgIyBMYXN0IHRoaW5nIE1BRCB3cml0ZXMuIEFmdGVyIHRoaXMsIGRhdGEgcGFzc2VzIHRvIENTRS4KICAgICMgdl9sYWJlbCAoMS4wLzAuNS8wLjApIGlzIHdoYXQgdGhlIGZlZWRiYWNrIGxvb3AgdXNlcyBmb3IgQnJpZXIgcmV3YXJkLgogICAgcG9vbF9pZHMgPSBbYy5jaHVua19pZCBmb3IgYyBpbiBldmlkZW5jZV9wb29sXQogICAgc3RvcmFnZS53cml0ZV9qdWRnZV92ZXJkaWN0cygKICAgICAgICBxdWVyeV9pZD1xdWVyeV9pZCwKICAgICAgICByb2xsb3V0X2lkPXJvbGxvdXRfaWQsCiAgICAgICAganVkZ2VfdmVyZGljdHM9anVkZ2VfdmVyZGljdHMsCiAgICAgICAgZXZpZGVuY2VfcG9vbF9pZHM9cG9vbF9pZHMsCiAgICApCgogICAgIyDilIDilIAgOC4gQ09NUFVURSBST1VUSU5HIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAogICAgcHJpbnQoIlxuW1N0ZXAgNC80XSBSb3V0aW5nIGRlY2lzaW9uLi4uIikKICAgIHJvdXRpbmcsIGFnZ3JlZ2F0ZSA9IF9jb21wdXRlX3JvdXRpbmcoZmluYWxfY2xhaW1zLCBqdWRnZV92ZXJkaWN0cykKICAgIHByaW50KGYiICAgICAgICAgICBBZ2dyZWdhdGUgY29uZmlkZW5jZToge2FnZ3JlZ2F0ZTouNGZ9IikKICAgIHByaW50KGYiICAgICAgICAgICBSb3V0aW5nIGRlY2lzaW9uICAgIDoge3JvdXRpbmd9IikKICAgIGlmIGNvcnJlY3Rpb25fc2lnbmFsOgogICAgICAgIHByaW50KGYiICAgICAgICAgICBDb3JyZWN0aW9uIHNpZ25hbCAgIDoge2NvcnJlY3Rpb25fc2lnbmFsWzo5MF19Li4uIikKCiAgICAjIOKUgOKUgCA5LiBVUERBVEUgUVVFUlkgV0lUSCBGSU5BTCBSRVNVTFRTIChUQUJMRSAxKSDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKICAgIHN0b3JhZ2UudXBkYXRlX3F1ZXJ5X2NzZSgKICAgICAgICBxdWVyeV9pZD1xdWVyeV9pZCwKICAgICAgICByb2xsb3V0X2lkPXJvbGxvdXRfaWQsCiAgICAgICAgY3NlX3Njb3JlPWFnZ3JlZ2F0ZSwKICAgICAgICByb3V0aW5nX2RlY2lzaW9uPXJvdXRpbmcsCiAgICApCgogICAgIyDilIDilIAgMTAuIEJVSUxEIFRSQU5TQ1JJUFQgKyBSRVRVUk4g4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACiAgICB0cmFuc2NyaXB0ID0gYnVpbGRfdHJhbnNjcmlwdCgKICAgICAgICBxdWVyeT1xdWVyeSwKICAgICAgICBsbG1fYW5zd2VyPWxsbV9hbnN3ZXIsCiAgICAgICAgY3ljbGVzPWN5Y2xlcywKICAgICAgICBqdWRnZV92ZXJkaWN0cz1qdWRnZV92ZXJkaWN0cywKICAgICAgICBjb3JyZWN0aW9uX3NpZ25hbD1jb3JyZWN0aW9uX3NpZ25hbCBvciAiIiwKICAgICkKCiAgICByZXR1cm4gTUFET3V0cHV0KAogICAgICAgIHF1ZXJ5PXF1ZXJ5LAogICAgICAgIGxsbV9hbnN3ZXI9bGxtX2Fuc3dlciwKICAgICAgICBjbGFpbXM9ZmluYWxfY2xhaW1zLAogICAgICAgIGRlYmF0ZV9jeWNsZXM9Y3ljbGVzLAogICAgICAgIGV2aWRlbmNlX3Bvb2w9ZXZpZGVuY2VfcG9vbCwKICAgICAgICBqdWRnZV92ZXJkaWN0cz1qdWRnZV92ZXJkaWN0cywKICAgICAgICBjb3JyZWN0aW9uX3NpZ25hbD1jb3JyZWN0aW9uX3NpZ25hbCwKICAgICAgICByb3V0aW5nX2RlY2lzaW9uPXJvdXRpbmcsCiAgICAgICAgYWdncmVnYXRlX2NvbmZpZGVuY2U9YWdncmVnYXRlLAogICAgICAgIGRlYmF0ZV90cmFuc2NyaXB0PXRyYW5zY3JpcHQsCiAgICAgICAgcXVlcnlfaWQ9cXVlcnlfaWQsCiAgICAgICAgcm9sbG91dF9pZD1yb2xsb3V0X2lkLAogICAgKQoKCiMg4pSA4pSAIFJvdXRpbmcgbG9naWMg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACgpkZWYgX2NvbXB1dGVfcm91dGluZygKICAgIGZpbmFsX2NsYWltczogICBMaXN0W0NsYWltXSwKICAgIGp1ZGdlX3ZlcmRpY3RzOiBMaXN0W0p1ZGdlVmVyZGljdF0sCikgLT4gVHVwbGVbc3RyLCBmbG9hdF06CiAgICAiIiIKICAgIFJldHVybnMgKHJvdXRpbmdfZGVjaXNpb24sIGFnZ3JlZ2F0ZV9jb25maWRlbmNlKS4KCiAgICBIYXJkIHJ1bGUgY2hlY2tlZCBmaXJzdCDigJQgb3ZlcnJpZGVzIGFnZ3JlZ2F0ZSBzY29yZSBlbnRpcmVseS4KICAgICIiIgogICAgY2xhaW1fbWFwID0ge2MuY2xhaW1faWQ6IGMgZm9yIGMgaW4gZmluYWxfY2xhaW1zfQogICAgYWdnICAgICAgID0gX2FnZ3JlZ2F0ZV9zY29yZShqdWRnZV92ZXJkaWN0cykKCiAgICAjIOKUgOKUgCBIYXJkIHJ1bGU6IGlzX21hdGVyaWFsIGNsYWltIHdpdGggdj0wLjAg4oaSIEhBUkRfQkxPQ0sg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACiAgICAjIEEgZmFicmljYXRlZCByZWd1bGF0b3J5IGNsYWltICh2PTAuMCkgdGhhdCBpcyBpc19tYXRlcmlhbCBjYW5ub3QKICAgICMgcmVhY2ggdGhlIHVzZXIgdW5kZXIgQU5ZIGNvbmZpZGVuY2UgdGhyZXNob2xkLiBObyByZXRyeS4gQmxvY2sgaW1tZWRpYXRlbHkuCiAgICBmb3IganYgaW4ganVkZ2VfdmVyZGljdHM6CiAgICAgICAgY2xhaW0gPSBjbGFpbV9tYXAuZ2V0KGp2LmNsYWltX2lkKQogICAgICAgIGlmIGNsYWltIGFuZCBjbGFpbS5pc19tYXRlcmlhbCBhbmQganYuc2NvcmUgPT0gMC4wOgogICAgICAgICAgICBwcmludChmIiAg4puUIEhBUkQgQkxPQ0sg4oCUIEN7anYuY2xhaW1faWR9IGlzX21hdGVyaWFsICsgdj0wLjA6ICIKICAgICAgICAgICAgICAgICAgZiJcIntqdi5jbGFpbV90ZXh0Wzo2MF19XCIiKQogICAgICAgICAgICByZXR1cm4gIkhBUkRfQkxPQ0siLCBhZ2cKCiAgICAjIOKUgOKUgCBTb2Z0IHJ1bGVzIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAogICAgaWYgYWdnID49IENPTkZJREVOQ0VfVEhSRVNIT0xEX0hJR0g6CiAgICAgICAgcmV0dXJuICJERUxJVkVSIiwgYWdnCiAgICBlbGlmIGFnZyA+PSBDT05GSURFTkNFX1RIUkVTSE9MRF9MT1c6CiAgICAgICAgcmV0dXJuICJSRVRSWSIsIGFnZwogICAgZWxzZToKICAgICAgICByZXR1cm4gIkhVTUFOX1JFVklFVyIsIGFnZwoKCmRlZiBfYWdncmVnYXRlX3Njb3JlKGp1ZGdlX3ZlcmRpY3RzOiBMaXN0W0p1ZGdlVmVyZGljdF0pIC0+IGZsb2F0OgogICAgIiIiCiAgICBXZWlnaHRlZCBtaW4tbWVhbiBhZ2dyZWdhdGU6CiAgICAgIE1hdGVyaWFsIGNsYWltczogICAgIG1pbiAod2Vha2VzdCBsaW5rIOKAlCBzYWZldHkgY3JpdGljYWwpCiAgICAgIE5vbi1tYXRlcmlhbCBjbGFpbXM6IG1lYW4KICAgICAgQ29tYmluZWQ6ICAgICAgICAgICAgNzAlIG1hdGVyaWFsIG1pbiArIDMwJSBub24tbWF0ZXJpYWwgbWVhbgogICAgIiIiCiAgICBpZiBub3QganVkZ2VfdmVyZGljdHM6CiAgICAgICAgcmV0dXJuIDAuNQoKICAgIG1hdCAgPSBbanYgZm9yIGp2IGluIGp1ZGdlX3ZlcmRpY3RzIGlmIGp2LmlzX21hdGVyaWFsXQogICAgbm1hdCA9IFtqdiBmb3IganYgaW4ganVkZ2VfdmVyZGljdHMgaWYgbm90IGp2LmlzX21hdGVyaWFsXQoKICAgIGlmIG1hdCBhbmQgbm1hdDoKICAgICAgICByZXR1cm4gcm91bmQoMC43MCAqIG1pbihqdi5zY29yZSBmb3IganYgaW4gbWF0KQogICAgICAgICAgICAgICAgICAgICArIDAuMzAgKiBzdW0oanYuc2NvcmUgZm9yIGp2IGluIG5tYXQpIC8gbGVuKG5tYXQpLCA0KQogICAgaWYgbWF0OgogICAgICAgIHJldHVybiByb3VuZChtaW4oanYuc2NvcmUgZm9yIGp2IGluIG1hdCksIDQpCiAgICByZXR1cm4gcm91bmQoc3VtKGp2LnNjb3JlIGZvciBqdiBpbiBubWF0KSAvIGxlbihubWF0KSwgNCkKCgojIOKUgOKUgCBQcmludCBoZWxwZXJzIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAoKZGVmIF9iYW5uZXIodGl0bGU6IHN0cikgLT4gTm9uZToKICAgIHByaW50KCJcbiIgKyAi4paTIiAqIDY4KQogICAgcHJpbnQoZiIgIEdVQVJEUkFJTFMgR0FURVdBWSDigJQge3RpdGxlfSIpCiAgICBwcmludCgi4paTIiAqIDY4KQoKCmRlZiBfcHJpbnRfanVkZ2VfdmVyZGljdHModmVyZGljdHM6IExpc3RbSnVkZ2VWZXJkaWN0XSkgLT4gTm9uZToKICAgIGljb24gPSB7MS4wOiAi4pyFIiwgMC41OiAi4pqg77iPICIsIDAuMDogIuKdjCJ9CiAgICBmb3IganYgaW4gdmVyZGljdHM6CiAgICAgICAgbWF0ID0gIuKaoCBtYXRlcmlhbCIgaWYganYuaXNfbWF0ZXJpYWwgZWxzZSAiICBjb250ZXh0ICIKICAgICAgICBwcmludChmIiAgICB7aWNvbi5nZXQoanYuc2NvcmUsJz8nKX0gW3ttYXR9XSBDe2p2LmNsYWltX2lkfTogIgogICAgICAgICAgICAgIGYidj17anYuc2NvcmV9ICBcIntqdi5jbGFpbV90ZXh0Wzo2MF19XCIiKQo=",
    "multi_agent/judge.py": "IiIiCmp1ZGdlLnB5IOKAlCBKdWRnZSBBZ2VudCAoUGFydGlhbGx5IEJsaW5kKQoKU0VFUzoKICAtIFVzZXIgcXVlcnkKICAtIEFnZW50IEEncyBmaW5hbCByZXZpc2VkIHZlcmRpY3RzICh2ZXJkaWN0IHRleHQgKyByZWFzb25pbmcpCiAgLSBFdmlkZW5jZSBwb29sIChhbGwgY2h1bmtzIGZyb20gYm90aCBhZ2VudHMg4oCUIHVubGFiZWxsZWQsIG5vIGFnZW50IGF0dHJpYnV0aW9uKQoKRE9FUyBOT1QgU0VFOgogIC0gQ29uZmlkZW5jZSBzY29yZXMgKGludGVudGlvbmFsbHkgc3RyaXBwZWQg4oCUIGF2b2lkcyBhbmNob3JpbmcpCiAgLSBBZ2VudCBCJ3MgY2hhbGxlbmdlIHRleHQgb3IgYXJndW1lbnRhdGl2ZSBmcmFtaW5nCiAgLSBXaGljaCBhZ2VudCByZXRyaWV2ZWQgd2hpY2ggY2h1bmsKClNjb3JlcyBlYWNoIG1hdGVyaWFsIGNsYWltIGluZGVwZW5kZW50bHk6IHYgPSAxLjAgLyAwLjUgLyAwLjAKR2VuZXJhdGVzIGEgY29ycmVjdGlvbl9zaWduYWwgZm9yIHRoZSByZXRyeSBsb29wIHdoZW4gY2xhaW1zIGFyZSB3cm9uZy4KIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBqc29uCmltcG9ydCByZQpmcm9tIHR5cGluZyBpbXBvcnQgTGlzdCwgT3B0aW9uYWwsIFR1cGxlCgpmcm9tIG9wZW5haSBpbXBvcnQgT3BlbkFJCgpmcm9tIG11bHRpX2FnZW50LmNvbmZpZyBpbXBvcnQgKAogICAgT0xMQU1BX0JBU0VfVVJMLCBPTExBTUFfQVBJX0tFWSwKICAgIEpVREdFX01PREVMLCBKVURHRV9QUk9WSURFUiwgQU5USFJPUElDX0FQSV9LRVksCikKZnJvbSBtdWx0aV9hZ2VudC5tb2RlbHMgaW1wb3J0IENsYWltLCBFdmlkZW5jZUNodW5rLCBKdWRnZVZlcmRpY3QKCiMg4pSA4pSAIENsaWVudCBzZXR1cCDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKX29sbGFtYV9jbGllbnQgPSBPcGVuQUkoYmFzZV91cmw9T0xMQU1BX0JBU0VfVVJMLCBhcGlfa2V5PU9MTEFNQV9BUElfS0VZKQoKX2FudGhyb3BpY19jbGllbnQgPSBOb25lCmlmIEpVREdFX1BST1ZJREVSID09ICJhbnRocm9waWMiOgogICAgdHJ5OgogICAgICAgIGltcG9ydCBhbnRocm9waWMgYXMgX2FudGhyb3BpY19zZGsKICAgICAgICBfYW50aHJvcGljX2NsaWVudCA9IF9hbnRocm9waWNfc2RrLkFudGhyb3BpYyhhcGlfa2V5PUFOVEhST1BJQ19BUElfS0VZKQogICAgICAgIHByaW50KGYiW0p1ZGdlXSBVc2luZyBDbGF1ZGUgdmlhIEFudGhyb3BpYyBBUEkg4oCUIG1vZGVsOiB7SlVER0VfTU9ERUx9IikKICAgIGV4Y2VwdCBJbXBvcnRFcnJvcjoKICAgICAgICBwcmludCgiW0p1ZGdlXSBXYXJuaW5nOiBhbnRocm9waWMgU0RLIG5vdCBpbnN0YWxsZWQuIEZhbGxpbmcgYmFjayB0byBPbGxhbWEuIikKCl9TWVNURU0gPSAoCiAgICAiWW91IGFyZSBhbiBpbXBhcnRpYWwgcmVndWxhdG9yeSBjb21wbGlhbmNlIGp1ZGdlLiAiCiAgICAiRXZhbHVhdGUgY2xhaW1zIHN0cmljdGx5IGJhc2VkIG9uIHRoZSBwcm92aWRlZCBldmlkZW5jZS4gIgogICAgIkJlIHByZWNpc2UgYW5kIGNpdGUgc3BlY2lmaWMgZXZpZGVuY2UgY2h1bmtzLiAiCiAgICAiRG8gbm90IHNwZWN1bGF0ZSBiZXlvbmQgd2hhdCB0aGUgZXZpZGVuY2Ugc3VwcG9ydHMuIgopCgpfSlVER0VfUFJPTVBUID0gIiIiXApZb3UgYXJlIGFuIGltcGFydGlhbCBKdWRnZSBldmFsdWF0aW5nIHdoZXRoZXIgYW4gQUktZ2VuZXJhdGVkIHJlZ3VsYXRvcnkgY29tcGxpYW5jZSBhbnN3ZXIgaXMgYWNjdXJhdGUuCgpZb3VyIHRhc2s6IGluZGVwZW5kZW50bHkgc2NvcmUgZWFjaCBjbGFpbSBhZ2FpbnN0IHRoZSBwcm92aWRlZCBldmlkZW5jZSBwb29sLgoKU0NPUklORzoKICAxLjAg4oCUIGNsYWltIGlzIGZ1bGx5IGFjY3VyYXRlIGFuZCBkaXJlY3RseSBzdXBwb3J0ZWQgYnkgdGhlIGV2aWRlbmNlCiAgMC41IOKAlCBjbGFpbSBpcyBwYXJ0aWFsbHkgY29ycmVjdCBidXQgbWlzc2luZyBpbXBvcnRhbnQgbnVhbmNlLCBjYXZlYXRzLCBvciBzY29wZSBxdWFsaWZpY2F0aW9ucwogIDAuMCDigJQgY2xhaW0gaXMgaW5hY2N1cmF0ZSwgdW5zdXBwb3J0ZWQsIG9yIGFwcGVhcnMgdG8gYmUgaGFsbHVjaW5hdGVkCgpJTlNUUlVDVElPTlM6Ci0gU2NvcmUgQUxMIGNsYWltcy4gRm9yIG5vbi1tYXRlcmlhbCBjbGFpbXMgKGlzX21hdGVyaWFsPWZhbHNlKTogYXNzaWduIHNjb3JlIGJhc2VkIG9uIGZhY3R1YWwgYWNjdXJhY3kuCi0gQ2l0ZSBzcGVjaWZpYyBjaHVua19pZHMgaW4geW91ciByZWFzb25pbmcuCi0gQWZ0ZXIgc2NvcmluZywgcHJvZHVjZSBhIGNvcnJlY3Rpb25fc2lnbmFsOgogICAgLSBBIGNsZWFyLCBhY3Rpb25hYmxlIGV4cGxhbmF0aW9uIG9mIHdoYXQgdGhlIG9yaWdpbmFsIGFuc3dlciBnb3Qgd3JvbmcgYW5kIHdoYXQgdGhlIGNvcnJlY3QgcmVndWxhdG9yeSBwb3NpdGlvbiBpcy4KICAgIC0gVGhpcyBzaWduYWwgd2lsbCBiZSBzZW50IHRvIHRoZSBMTE0gb24gcmV0cnkgc28gaXQgbXVzdCBiZSBzZWxmLWNvbnRhaW5lZCBhbmQgcHJlY2lzZS4KICAgIC0gSWYgQUxMIG1hdGVyaWFsIGNsYWltcyBzY29yZSAxLjA6IHNldCBjb3JyZWN0aW9uX3NpZ25hbCB0byBudWxsLgoKVXNlciBxdWVyeToKe3F1ZXJ5fQoKQ2xhaW1zIHRvIGV2YWx1YXRlIChmcm9tIHZlcmlmaWNhdGlvbiBhZ2VudCDigJQgY29uZmlkZW5jZSBzY29yZXMgUkVEQUNURUQpOgp7Y2xhaW1zX2pzb259CgpFdmlkZW5jZSBwb29sIChyZWd1bGF0b3J5IGNodW5rcyBmcm9tIGFsbCBzb3VyY2VzIOKAlCBldmFsdWF0ZSBhZ2FpbnN0IHRoZXNlKToKe2V2aWRlbmNlX3Bvb2xfanNvbn0KClJldHVybiBPTkxZIHZhbGlkIEpTT04uIE5vIG1hcmtkb3duIGZlbmNlcywgbm8gZXhwbGFuYXRpb24gb3V0c2lkZSB0aGUgSlNPTi4KT3V0cHV0IHNjaGVtYSAocmVwbGFjZSBhbGwgcGxhY2Vob2xkZXIgdmFsdWVzIHdpdGggeW91ciBhY3R1YWwgZXZhbHVhdGlvbik6Cgp7ewogICJ2ZXJkaWN0cyI6IFsKICAgIHt7CiAgICAgICJjbGFpbV9pZCI6IDxpbnRlZ2VyIG1hdGNoaW5nIGEgY2xhaW1faWQgYWJvdmU+LAogICAgICAic2NvcmUiOiA8MS4wIHwgMC41IHwgMC4wPiwKICAgICAgInJlYXNvbmluZyI6ICI8Y2l0ZSB0aGUgc3BlY2lmaWMgY2h1bmtfaWQocykgZnJvbSB0aGUgZXZpZGVuY2UgcG9vbCB0aGF0IHN1cHBvcnQgeW91ciBzY29yZSBhbmQgZXhwbGFpbiB3aHk+IgogICAgfX0KICBdLAogICJjb3JyZWN0aW9uX3NpZ25hbCI6ICI8Y29uY2lzZSBhY3Rpb25hYmxlIGRlc2NyaXB0aW9uIG9mIHdoYXQgdGhlIG9yaWdpbmFsIGFuc3dlciBnb3Qgd3JvbmcgYW5kIHdoYXQgdGhlIGNvcnJlY3QgcmVndWxhdG9yeSBwb3NpdGlvbiBpcywgZ3JvdW5kZWQgaW4gdGhlIGV2aWRlbmNlIHBvb2wg4oCUIG9yIG51bGwgaWYgYWxsIG1hdGVyaWFsIGNsYWltcyBzY29yZSAxLjA+Igp9fQoiIiIKCgpkZWYganVkZ2VfY2xhaW1zKAogICAgcXVlcnk6ICAgICAgICAgc3RyLAogICAgZmluYWxfY2xhaW1zOiAgTGlzdFtDbGFpbV0sCiAgICBldmlkZW5jZV9wb29sOiBMaXN0W0V2aWRlbmNlQ2h1bmtdLAopIC0+IFR1cGxlW0xpc3RbSnVkZ2VWZXJkaWN0XSwgT3B0aW9uYWxbc3RyXV06CiAgICAiIiIKICAgIEp1ZGdlIGV2YWx1YXRlcyBhbGwgY2xhaW1zIGFnYWluc3QgdGhlIGV2aWRlbmNlIHBvb2wuCgogICAgSW50ZW50aW9uYWxseSBTVFJJUFMgY29uZmlkZW5jZSBzY29yZXMgZnJvbSB0aGUgY2xhaW1zIEpTT04KICAgIHNvIHRoZSBKdWRnZSBzY29yZXMgaW5kZXBlbmRlbnRseSB3aXRob3V0IGFuY2hvcmluZy4KCiAgICBSZXR1cm5zOiAoanVkZ2VfdmVyZGljdHMsIGNvcnJlY3Rpb25fc2lnbmFsKQogICAgIiIiCiAgICAjIEJ1aWxkIGNsYWltcyBKU09OIHdpdGggY29uZmlkZW5jZSBTVFJJUFBFRAogICAgY2xhaW1zX2pzb24gPSBqc29uLmR1bXBzKAogICAgICAgIFt7CiAgICAgICAgICAgICJjbGFpbV9pZCI6ICAgICAgYy5jbGFpbV9pZCwKICAgICAgICAgICAgImNsYWltX3RleHQiOiAgICBjLmNsYWltX3RleHQsCiAgICAgICAgICAgICJpc19tYXRlcmlhbCI6ICAgYy5pc19tYXRlcmlhbCwKICAgICAgICAgICAgImFnZW50X3ZlcmRpY3QiOiBjLnZlcmRpY3QudmFsdWUgaWYgYy52ZXJkaWN0IGVsc2UgIklESyIsCiAgICAgICAgICAgICJhZ2VudF9yZWFzb25pbmciOiBjLnJlYXNvbmluZywKICAgICAgICAgICAgIyBjb25maWRlbmNlIGludGVudGlvbmFsbHkgb21pdHRlZAogICAgICAgIH0gZm9yIGMgaW4gZmluYWxfY2xhaW1zXSwKICAgICAgICBpbmRlbnQ9MiwKICAgICkKCiAgICBldmlkZW5jZV9qc29uID0ganNvbi5kdW1wcygKICAgICAgICBbeyJjaHVua19pZCI6IGMuY2h1bmtfaWQsICJzb3VyY2UiOiBjLnNvdXJjZSwgInRpZXIiOiBjLnRpZXIsICJ0ZXh0IjogYy50ZXh0fQogICAgICAgICBmb3IgYyBpbiBldmlkZW5jZV9wb29sXSwKICAgICAgICBpbmRlbnQ9MiwKICAgICkKCiAgICBwcm9tcHQgPSBfSlVER0VfUFJPTVBULmZvcm1hdCgKICAgICAgICBxdWVyeT1xdWVyeS5zdHJpcCgpLAogICAgICAgIGNsYWltc19qc29uPWNsYWltc19qc29uLAogICAgICAgIGV2aWRlbmNlX3Bvb2xfanNvbj1ldmlkZW5jZV9qc29uLAogICAgKQoKICAgIHJhdyA9IF9jYWxsX2p1ZGdlKHByb21wdCkKCiAgICB0cnk6CiAgICAgICAgZGF0YSA9IGpzb24ubG9hZHMocmF3KQogICAgZXhjZXB0IGpzb24uSlNPTkRlY29kZUVycm9yIGFzIGU6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigKICAgICAgICAgICAgZiJbanVkZ2VdIEZhaWxlZCB0byBwYXJzZSBKdWRnZSByZXNwb25zZS5cbkVycm9yOiB7ZX1cblJhdzpcbntyYXdbOjUwMF19IgogICAgICAgICkKCiAgICBjb3JyZWN0aW9uX3NpZ25hbDogT3B0aW9uYWxbc3RyXSA9IGRhdGEuZ2V0KCJjb3JyZWN0aW9uX3NpZ25hbCIpIG9yIE5vbmUKCiAgICB2ZXJkaWN0X21hcCA9IHt2WyJjbGFpbV9pZCJdOiB2IGZvciB2IGluIGRhdGEuZ2V0KCJ2ZXJkaWN0cyIsIFtdKX0KICAgIGp1ZGdlX3ZlcmRpY3RzOiBMaXN0W0p1ZGdlVmVyZGljdF0gPSBbXQoKICAgIGZvciBjbGFpbSBpbiBmaW5hbF9jbGFpbXM6CiAgICAgICAgdiA9IHZlcmRpY3RfbWFwLmdldChjbGFpbS5jbGFpbV9pZCwge30pCiAgICAgICAgc2NvcmUgPSBmbG9hdCh2LmdldCgic2NvcmUiLCAxLjAgaWYgbm90IGNsYWltLmlzX21hdGVyaWFsIGVsc2UgMC41KSkKICAgICAgICAjIENsYW1wIHRvIHZhbGlkIHZhbHVlcwogICAgICAgIGlmIHNjb3JlID4gMC43NToKICAgICAgICAgICAgc2NvcmUgPSAxLjAKICAgICAgICBlbGlmIHNjb3JlID4gMC4yNToKICAgICAgICAgICAgc2NvcmUgPSAwLjUKICAgICAgICBlbHNlOgogICAgICAgICAgICBzY29yZSA9IDAuMAoKICAgICAgICBqdWRnZV92ZXJkaWN0cy5hcHBlbmQoSnVkZ2VWZXJkaWN0KAogICAgICAgICAgICBjbGFpbV9pZD1jbGFpbS5jbGFpbV9pZCwKICAgICAgICAgICAgY2xhaW1fdGV4dD1jbGFpbS5jbGFpbV90ZXh0LAogICAgICAgICAgICBpc19tYXRlcmlhbD1jbGFpbS5pc19tYXRlcmlhbCwKICAgICAgICAgICAgc2NvcmU9c2NvcmUsCiAgICAgICAgICAgIHJlYXNvbmluZz12LmdldCgicmVhc29uaW5nIiwgIiIpLAogICAgICAgICkpCgogICAgcmV0dXJuIGp1ZGdlX3ZlcmRpY3RzLCBjb3JyZWN0aW9uX3NpZ25hbAoKCiMg4pSA4pSAIEhlbHBlcnMg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACgpkZWYgX2NhbGxfanVkZ2UocHJvbXB0OiBzdHIpIC0+IHN0cjoKICAgICIiIgogICAgQ2FsbCB0aGUganVkZ2UgTExNIGFuZCByZXR1cm4gcmF3IHRleHQgcmVzcG9uc2UuCiAgICBSb3V0ZXMgdG8gQW50aHJvcGljIENsYXVkZSBpZiBKVURHRV9QUk9WSURFUj1hbnRocm9waWMsIGVsc2UgT2xsYW1hLgogICAgIiIiCiAgICBpZiBKVURHRV9QUk9WSURFUiA9PSAiYW50aHJvcGljIiBhbmQgX2FudGhyb3BpY19jbGllbnQgaXMgbm90IE5vbmU6CiAgICAgICAgbWVzc2FnZSA9IF9hbnRocm9waWNfY2xpZW50Lm1lc3NhZ2VzLmNyZWF0ZSgKICAgICAgICAgICAgbW9kZWw9SlVER0VfTU9ERUwsCiAgICAgICAgICAgIG1heF90b2tlbnM9MTAyNCwKICAgICAgICAgICAgdGVtcGVyYXR1cmU9MC4wLAogICAgICAgICAgICBzeXN0ZW09X1NZU1RFTSwKICAgICAgICAgICAgbWVzc2FnZXM9W3sicm9sZSI6ICJ1c2VyIiwgImNvbnRlbnQiOiBwcm9tcHR9XSwKICAgICAgICApCiAgICAgICAgcmV0dXJuIF9jbGVhbl9qc29uX29iamVjdChtZXNzYWdlLmNvbnRlbnRbMF0udGV4dCkKCiAgICAjIEZhbGxiYWNrOiBPbGxhbWEKICAgIHJlc3BvbnNlID0gX29sbGFtYV9jbGllbnQuY2hhdC5jb21wbGV0aW9ucy5jcmVhdGUoCiAgICAgICAgbW9kZWw9SlVER0VfTU9ERUwsCiAgICAgICAgbWVzc2FnZXM9WwogICAgICAgICAgICB7InJvbGUiOiAic3lzdGVtIiwgImNvbnRlbnQiOiBfU1lTVEVNfSwKICAgICAgICAgICAgeyJyb2xlIjogInVzZXIiLCAgICJjb250ZW50IjogcHJvbXB0fSwKICAgICAgICBdLAogICAgICAgIHRlbXBlcmF0dXJlPTAuMCwKICAgICkKICAgIHJldHVybiBfY2xlYW5fanNvbl9vYmplY3QocmVzcG9uc2UuY2hvaWNlc1swXS5tZXNzYWdlLmNvbnRlbnQpCgoKZGVmIF9jbGVhbl9qc29uX29iamVjdChyYXc6IHN0cikgLT4gc3RyOgogICAgIiIiU3RyaXAgbWFya2Rvd24gZmVuY2VzIGFuZCBpc29sYXRlIHRoZSBKU09OIG9iamVjdC4iIiIKICAgIHJhdyA9IHJlLnN1YihyImBgYGpzb25ccyoiLCAiIiwgcmF3KQogICAgcmF3ID0gcmUuc3ViKHIiYGBgXHMqIiwgICAgICIiLCByYXcpCiAgICByYXcgPSByYXcuc3RyaXAoKQogICAgIyBQcmVmZXIgb3V0ZXJtb3N0IHsgLi4uIH0KICAgIG1hdGNoID0gcmUuc2VhcmNoKHIiXHsuKlx9IiwgcmF3LCByZS5ET1RBTEwpCiAgICByZXR1cm4gbWF0Y2guZ3JvdXAoKSBpZiBtYXRjaCBlbHNlIHJhdwo=",
    "multi_agent/config.py": "IiIiCmNvbmZpZy5weSDigJQgY2VudHJhbCBjb25maWd1cmF0aW9uIGZvciB0aGUgTUFEIHBpcGVsaW5lLgpBbGwgdmFsdWVzIGFyZSBvdmVycmlkYWJsZSB2aWEgZW52aXJvbm1lbnQgdmFyaWFibGVzLgoiIiIKaW1wb3J0IG9zCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKIyDilIDilIAgTExNIChBZ2VudCBBICsgQWdlbnQgQiB2aWEgSHVnZ2luZ0ZhY2UgdHJhbnNmb3JtZXJzKSDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKIyBIdWdnaW5nRmFjZSBtb2RlbCBJRCBvciBsb2NhbCBwYXRoIHRvIHRoZSBjaGF0IG1vZGVsIHVzZWQgYnkgYWdlbnRzLgojIERlZmF1bHQ6IFF3ZW4vUXdlbjIuNS03Qi1JbnN0cnVjdAojIEFsdGVybmF0aXZlczogUXdlbi9Rd2VuMi41LTNCLUluc3RydWN0IChzbWFsbGVyLCBmYXN0ZXIsIGxvd2VyIFZSQU0pCkFHRU5UX01PREVMOiBzdHIgPSBvcy5nZXRlbnYoIkFHRU5UX01PREVMIiwgIlF3ZW4vUXdlbjIuNS03Qi1JbnN0cnVjdCIpCgojIOKUgOKUgCBKdWRnZSBtb2RlbCDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKIyBKVURHRV9QUk9WSURFUj1hbnRocm9waWMg4oaSIHVzZXMgQ2xhdWRlIHZpYSBBbnRocm9waWMgU0RLIChyZWNvbW1lbmRlZCkKIyBKVURHRV9QUk9WSURFUj1vbGxhbWEgICAg4oaSIHVzZXMgT2xsYW1hIG1vZGVsIChmYWxsYmFjayAvIG9mZmxpbmUpCkpVREdFX1BST1ZJREVSOiAgICAgIHN0ciA9IG9zLmdldGVudigiSlVER0VfUFJPVklERVIiLCAiYW50aHJvcGljIikKSlVER0VfTU9ERUw6ICAgICAgICAgc3RyID0gb3MuZ2V0ZW52KCJKVURHRV9NT0RFTCIsICJjbGF1ZGUtaGFpa3UtNC01LTIwMjUxMDAxIikKQU5USFJPUElDX0FQSV9LRVk6ICAgc3RyID0gb3MuZ2V0ZW52KCJBTlRIUk9QSUNfQVBJX0tFWSIsICIiKQoKIyBLZWVwIHRoZXNlIGZvciBiYWNrd2FyZHMgY29tcGF0aWJpbGl0eSBpZiBqdWRnZSBmYWxscyBiYWNrIHRvIE9sbGFtYQpPTExBTUFfQkFTRV9VUkw6IHN0ciA9IG9zLmdldGVudigiT0xMQU1BX0JBU0VfVVJMIiwgImh0dHA6Ly9sb2NhbGhvc3Q6MTE0MzQvdjEiKQpPTExBTUFfQVBJX0tFWTogIHN0ciA9ICJvbGxhbWEiCgojIOKUgOKUgCBSQUcgLyBDaHVua3Mg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACl9yZXBvX3Jvb3QgPSBQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50LnBhcmVudApDSFVOS1NfSlNPTkxfUEFUSDogc3RyID0gb3MuZ2V0ZW52KAogICAgIkNIVU5LU19KU09OTF9QQVRIIiwKICAgIHN0cihfcmVwb19yb290IC8gInJhZyIgLyAiY2h1bmtzLmpzb25sIikKKQoKIyDilIDilIAgRGViYXRlIHNldHRpbmdzIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgApNQVhfQ1lDTEVTOiAgICAgICAgICAgICAgaW50ID0gaW50KG9zLmdldGVudigiTUFYX0NZQ0xFUyIsICIyIikpClRPUF9LX0NIVU5LUzogICAgICAgICAgICBpbnQgPSBpbnQob3MuZ2V0ZW52KCJUT1BfS19DSFVOS1MiLCAiNSIpKQpUT1BfS19DSEFMTEVOR0VfQ0hVTktTOiAgaW50ID0gaW50KG9zLmdldGVudigiVE9QX0tfQ0hBTExFTkdFX0NIVU5LUyIsICIzIikpCgojIOKUgOKUgCBSb3V0aW5nIHRocmVzaG9sZHMg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACkNPTkZJREVOQ0VfVEhSRVNIT0xEX0hJR0g6IGZsb2F0ID0gZmxvYXQob3MuZ2V0ZW52KCJDT05GSURFTkNFX1RIUkVTSE9MRF9ISUdIIiwgIjAuOCIpKQpDT05GSURFTkNFX1RIUkVTSE9MRF9MT1c6ICBmbG9hdCA9IGZsb2F0KG9zLmdldGVudigiQ09ORklERU5DRV9USFJFU0hPTERfTE9XIiwgIjAuNCIpKQoKIyDilIDilIAgU3RvcmFnZSAoU1FMaXRlKSDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKREJfUEFUSDogc3RyID0gb3MuZ2V0ZW52KCJNQURfREJfUEFUSCIsIHN0cihfcmVwb19yb290IC8gIm1hZF9zdG9yZS5kYiIpKQoKIyDilIDilIAgRmFzdEFQSSDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKQVBJX0hPU1Q6IHN0ciA9IG9zLmdldGVudigiTUFEX0FQSV9IT1NUIiwgIjAuMC4wLjAiKQpBUElfUE9SVDogaW50ID0gaW50KG9zLmdldGVudigiTUFEX0FQSV9QT1JUIiwgIjgwMDEiKSkK",
    "multi_agent/hf_client.py": "IiIiCmhmX2NsaWVudC5weSDigJQgSHVnZ2luZ0ZhY2UtbmF0aXZlIExMTSBjbGllbnQgZm9yIHRoZSBNQUQgcGlwZWxpbmUuCgpSZXBsYWNlcyBPbGxhbWEgYXMgdGhlIGJhY2tlbmQgZm9yIEFnZW50IEEsIEFnZW50IEIsIGFuZCBDbGFpbSBFeHRyYWN0b3IuClByb3ZpZGVzIGFuIE9wZW5BSS1jb21wYXRpYmxlIGludGVyZmFjZSBzbyBubyBjaGFuZ2VzIHRvIHRoZSBhZ2VudApjYWxsIHNpdGVzIGFyZSBuZWVkZWQuCgpNb2RlbDogUXdlbi9Rd2VuMi41LTdCLUluc3RydWN0IChkZWZhdWx0IOKAlCBvdmVycmlkZSB3aXRoIEFHRU5UX01PREVMIGVudiB2YXIpCiAgLSBiZmxvYXQxNiBmdWxsIHByZWNpc2lvbiBvbiBHUFUgd2l0aCA+PSAxOCBHQiBWUkFNCiAgLSA0LWJpdCBORjQgcXVhbnRpemVkIG9uIEdQVSB3aXRoIDwgMTggR0IgVlJBTSAodXNlcyBiaXRzYW5kYnl0ZXMpCiAgLSBmbG9hdDMyIG9uIENQVSAoc2xvdyDigJQgbm90IHJlY29tbWVuZGVkKQoKVlJBTSBndWlkZToKICBBMTAwIDQwLzgwIEdCIDogYmZsb2F0MTYsIGJvdGggTmVtb3Ryb24tOEIgKyBRd2VuLTdCIGZpdCBjb21mb3J0YWJseQogIFJUWCAzMDkwIDI0IEdCOiBiZmxvYXQxNiwgdGlnaHQgYnV0IHdvcmtzCiAgVDQgMTYgR0IgICAgICA6IDQtYml0IFF3ZW4tN0IgKH40IEdCKSArIE5lbW90cm9uLThCICh+MTUgR0IpID0gfjE5IEdCIC0+IHVzZSA0LWJpdAogIFQ0IDE2IEdCICAgICAgOiBDb25zaWRlciBRd2VuMi41LTNCLUluc3RydWN0IGlmIFZSQU0gaXMgdG9vIHRpZ2h0CiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgb3MKZnJvbSB0eXBpbmcgaW1wb3J0IExpc3QsIERpY3QKCmltcG9ydCB0b3JjaAoKX2hmX2NsaWVudCA9IE5vbmUgICAjIHByb2Nlc3Mtd2lkZSBzaW5nbGV0b24KCgpkZWYgZ2V0X2hmX2NsaWVudCgpIC0+ICJIRkNoYXRDbGllbnQiOgogICAgIiIiUmV0dXJuIHRoZSBzaW5nbGV0b24gSEZDaGF0Q2xpZW50LCBjcmVhdGluZyBpdCBvbiBmaXJzdCBjYWxsLiIiIgogICAgZ2xvYmFsIF9oZl9jbGllbnQKICAgIGlmIF9oZl9jbGllbnQgaXMgTm9uZToKICAgICAgICBtb2RlbF9uYW1lID0gb3MuZ2V0ZW52KCJBR0VOVF9NT0RFTCIsICJRd2VuL1F3ZW4yLjUtN0ItSW5zdHJ1Y3QiKQogICAgICAgIF9oZl9jbGllbnQgPSBIRkNoYXRDbGllbnQobW9kZWxfbmFtZSkKICAgIHJldHVybiBfaGZfY2xpZW50CgoKIyDilIDilIAgTW9jayBPcGVuQUkgcmVzcG9uc2Ugb2JqZWN0cyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKCmNsYXNzIF9NZXNzYWdlOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGNvbnRlbnQ6IHN0cik6CiAgICAgICAgc2VsZi5jb250ZW50ID0gY29udGVudAoKY2xhc3MgX0Nob2ljZToKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBjb250ZW50OiBzdHIpOgogICAgICAgIHNlbGYubWVzc2FnZSA9IF9NZXNzYWdlKGNvbnRlbnQpCgpjbGFzcyBfQ29tcGxldGlvblJlc3BvbnNlOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGNvbnRlbnQ6IHN0cik6CiAgICAgICAgc2VsZi5jaG9pY2VzID0gW19DaG9pY2UoY29udGVudCldCgoKIyDilIDilIAgT3BlbkFJLWNvbXBhdGlibGUgY29tcGxldGlvbnMgaW50ZXJmYWNlIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAoKY2xhc3MgX0NvbXBsZXRpb25zOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHBhcmVudDogIkhGQ2hhdENsaWVudCIpOgogICAgICAgIHNlbGYuX3BhcmVudCA9IHBhcmVudAoKICAgIGRlZiBjcmVhdGUoCiAgICAgICAgc2VsZiwKICAgICAgICBtb2RlbDogc3RyID0gIiIsICAgICAgICAgICMgYWNjZXB0ZWQgYnV0IGlnbm9yZWQg4oCUIG1vZGVsIGFscmVhZHkgbG9hZGVkCiAgICAgICAgbWVzc2FnZXM6IExpc3RbRGljdF0gPSBOb25lLAogICAgICAgIHRlbXBlcmF0dXJlOiBmbG9hdCA9IDAuMSwKICAgICAgICBtYXhfdG9rZW5zOiBpbnQgPSAyMDQ4LAogICAgICAgICoqa3dhcmdzLAogICAgKSAtPiBfQ29tcGxldGlvblJlc3BvbnNlOgogICAgICAgIHJldHVybiBzZWxmLl9wYXJlbnQuX2dlbmVyYXRlKG1lc3NhZ2VzIG9yIFtdLCB0ZW1wZXJhdHVyZSwgbWF4X3Rva2VucykKCgpjbGFzcyBfQ2hhdDoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBwYXJlbnQ6ICJIRkNoYXRDbGllbnQiKToKICAgICAgICBzZWxmLmNvbXBsZXRpb25zID0gX0NvbXBsZXRpb25zKHBhcmVudCkKCgojIOKUgOKUgCBNYWluIGNsaWVudCDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKCmNsYXNzIEhGQ2hhdENsaWVudDoKICAgICIiIgogICAgRHJvcC1pbiByZXBsYWNlbWVudCBmb3Igb3BlbmFpLk9wZW5BSSBiYWNrZWQgYnkgYSBsb2NhbCBIdWdnaW5nRmFjZSBtb2RlbC4KCiAgICBJbnRlcmZhY2UgKGlkZW50aWNhbCB0byBPcGVuQUkgY2xpZW50KToKICAgICAgICBjbGllbnQgPSBnZXRfaGZfY2xpZW50KCkKICAgICAgICByZXNwICAgPSBjbGllbnQuY2hhdC5jb21wbGV0aW9ucy5jcmVhdGUoCiAgICAgICAgICAgICAgICAgICAgIG1vZGVsPSIuLi4iLCAgICMgaWdub3JlZCDigJQgYWxyZWFkeSBsb2FkZWQKICAgICAgICAgICAgICAgICAgICAgbWVzc2FnZXM9W3sicm9sZSI6ICJzeXN0ZW0iLCAiY29udGVudCI6ICIuLi4ifSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHsicm9sZSI6ICJ1c2VyIiwgICAiY29udGVudCI6ICIuLi4ifV0sCiAgICAgICAgICAgICAgICAgICAgIHRlbXBlcmF0dXJlPTAuMSwKICAgICAgICAgICAgICAgICApCiAgICAgICAgdGV4dCA9IHJlc3AuY2hvaWNlc1swXS5tZXNzYWdlLmNvbnRlbnQKICAgICIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBtb2RlbF9uYW1lOiBzdHIpOgogICAgICAgIGZyb20gdHJhbnNmb3JtZXJzIGltcG9ydCBBdXRvVG9rZW5pemVyLCBBdXRvTW9kZWxGb3JDYXVzYWxMTQoKICAgICAgICBwcmludChmIltIRkNsaWVudF0gTG9hZGluZyB7bW9kZWxfbmFtZX0gLi4uIikKICAgICAgICBoZl90b2tlbiA9IG9zLmdldGVudigiSEZfVE9LRU4iKSBvciBOb25lCgogICAgICAgIHNlbGYuX3Rva2VuaXplciA9IEF1dG9Ub2tlbml6ZXIuZnJvbV9wcmV0cmFpbmVkKAogICAgICAgICAgICBtb2RlbF9uYW1lLAogICAgICAgICAgICB0cnVzdF9yZW1vdGVfY29kZT1UcnVlLAogICAgICAgICAgICB0b2tlbj1oZl90b2tlbiwKICAgICAgICApCgogICAgICAgICMgU2VsZWN0IHByZWNpc2lvbiBiYXNlZCBvbiBhdmFpbGFibGUgVlJBTSBzbyB0aGUgbW9kZWwgYW5kCiAgICAgICAgIyBOZW1vdHJvbi04QiAoMTUgR0IpIGNhbiBjb2V4aXN0IG9uIHRoZSBzYW1lIEdQVS4KICAgICAgICBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgICAgICAgICB2cmFtX2diID0gdG9yY2guY3VkYS5nZXRfZGV2aWNlX3Byb3BlcnRpZXMoMCkudG90YWxfbWVtb3J5IC8gMWU5CiAgICAgICAgICAgIGlmIHZyYW1fZ2IgPj0gMTg6CiAgICAgICAgICAgICAgICBwcmludChmIltIRkNsaWVudF0ge3ZyYW1fZ2I6LjBmfSBHQiBWUkFNIOKAlCBsb2FkaW5nIGJmbG9hdDE2IikKICAgICAgICAgICAgICAgIHNlbGYuX21vZGVsID0gQXV0b01vZGVsRm9yQ2F1c2FsTE0uZnJvbV9wcmV0cmFpbmVkKAogICAgICAgICAgICAgICAgICAgIG1vZGVsX25hbWUsCiAgICAgICAgICAgICAgICAgICAgdHJ1c3RfcmVtb3RlX2NvZGU9VHJ1ZSwKICAgICAgICAgICAgICAgICAgICB0b3JjaF9kdHlwZT10b3JjaC5iZmxvYXQxNiwKICAgICAgICAgICAgICAgICAgICBkZXZpY2VfbWFwPSJhdXRvIiwKICAgICAgICAgICAgICAgICAgICB0b2tlbj1oZl90b2tlbiwKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIHByaW50KGYiW0hGQ2xpZW50XSB7dnJhbV9nYjouMGZ9IEdCIFZSQU0g4oCUIGxvYWRpbmcgNC1iaXQgTkY0IChiaXRzYW5kYnl0ZXMpIikKICAgICAgICAgICAgICAgIGZyb20gdHJhbnNmb3JtZXJzIGltcG9ydCBCaXRzQW5kQnl0ZXNDb25maWcKICAgICAgICAgICAgICAgIGJuYiA9IEJpdHNBbmRCeXRlc0NvbmZpZygKICAgICAgICAgICAgICAgICAgICBsb2FkX2luXzRiaXQ9VHJ1ZSwKICAgICAgICAgICAgICAgICAgICBibmJfNGJpdF9jb21wdXRlX2R0eXBlPXRvcmNoLmJmbG9hdDE2LAogICAgICAgICAgICAgICAgICAgIGJuYl80Yml0X3VzZV9kb3VibGVfcXVhbnQ9VHJ1ZSwKICAgICAgICAgICAgICAgICAgICBibmJfNGJpdF9xdWFudF90eXBlPSJuZjQiLAogICAgICAgICAgICAgICAgKQogICAgICAgICAgICAgICAgc2VsZi5fbW9kZWwgPSBBdXRvTW9kZWxGb3JDYXVzYWxMTS5mcm9tX3ByZXRyYWluZWQoCiAgICAgICAgICAgICAgICAgICAgbW9kZWxfbmFtZSwKICAgICAgICAgICAgICAgICAgICB0cnVzdF9yZW1vdGVfY29kZT1UcnVlLAogICAgICAgICAgICAgICAgICAgIHF1YW50aXphdGlvbl9jb25maWc9Ym5iLAogICAgICAgICAgICAgICAgICAgIGRldmljZV9tYXA9ImF1dG8iLAogICAgICAgICAgICAgICAgICAgIHRva2VuPWhmX3Rva2VuLAogICAgICAgICAgICAgICAgKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHByaW50KCJbSEZDbGllbnRdIE5vIENVREEgR1BVIOKAlCBsb2FkaW5nIG9uIENQVSAoc2xvdykiKQogICAgICAgICAgICBzZWxmLl9tb2RlbCA9IEF1dG9Nb2RlbEZvckNhdXNhbExNLmZyb21fcHJldHJhaW5lZCgKICAgICAgICAgICAgICAgIG1vZGVsX25hbWUsCiAgICAgICAgICAgICAgICB0cnVzdF9yZW1vdGVfY29kZT1UcnVlLAogICAgICAgICAgICAgICAgdG9yY2hfZHR5cGU9dG9yY2guZmxvYXQzMiwKICAgICAgICAgICAgICAgIHRva2VuPWhmX3Rva2VuLAogICAgICAgICAgICApCgogICAgICAgIHNlbGYuX21vZGVsLmV2YWwoKQogICAgICAgIGRldmljZSA9IG5leHQoc2VsZi5fbW9kZWwucGFyYW1ldGVycygpKS5kZXZpY2UKICAgICAgICBwcmludChmIltIRkNsaWVudF0gUmVhZHkgb24ge2RldmljZX0iKQoKICAgICAgICBzZWxmLmNoYXQgPSBfQ2hhdChzZWxmKQoKICAgIGRlZiBfZ2VuZXJhdGUoCiAgICAgICAgc2VsZiwKICAgICAgICBtZXNzYWdlczogTGlzdFtEaWN0XSwKICAgICAgICB0ZW1wZXJhdHVyZTogZmxvYXQsCiAgICAgICAgbWF4X25ld190b2tlbnM6IGludCwKICAgICkgLT4gX0NvbXBsZXRpb25SZXNwb25zZToKICAgICAgICAjIEFwcGx5IHRoZSBtb2RlbCdzIGNoYXQgdGVtcGxhdGUgKGhhbmRsZXMgc3lzdGVtL3VzZXIvYXNzaXN0YW50IHR1cm5zKQogICAgICAgIHRleHQgPSBzZWxmLl90b2tlbml6ZXIuYXBwbHlfY2hhdF90ZW1wbGF0ZSgKICAgICAgICAgICAgbWVzc2FnZXMsCiAgICAgICAgICAgIHRva2VuaXplPUZhbHNlLAogICAgICAgICAgICBhZGRfZ2VuZXJhdGlvbl9wcm9tcHQ9VHJ1ZSwKICAgICAgICApCiAgICAgICAgaW5wdXRzID0gc2VsZi5fdG9rZW5pemVyKHRleHQsIHJldHVybl90ZW5zb3JzPSJwdCIpLnRvKHNlbGYuX21vZGVsLmRldmljZSkKCiAgICAgICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgICAgIG91dHB1dHMgPSBzZWxmLl9tb2RlbC5nZW5lcmF0ZSgKICAgICAgICAgICAgICAgICoqaW5wdXRzLAogICAgICAgICAgICAgICAgbWF4X25ld190b2tlbnM9bWF4X25ld190b2tlbnMsCiAgICAgICAgICAgICAgICB0ZW1wZXJhdHVyZT1tYXgoZmxvYXQodGVtcGVyYXR1cmUpLCAxZS03KSwKICAgICAgICAgICAgICAgIGRvX3NhbXBsZT0odGVtcGVyYXR1cmUgPiAwLjA1KSwKICAgICAgICAgICAgICAgIHBhZF90b2tlbl9pZD1zZWxmLl90b2tlbml6ZXIuZW9zX3Rva2VuX2lkLAogICAgICAgICAgICApCgogICAgICAgICMgRGVjb2RlIG9ubHkgdGhlIG5ldyB0b2tlbnMgKG5vdCB0aGUgcHJvbXB0KQogICAgICAgIG5ld190b2tlbnMgPSBvdXRwdXRzWzBdW2lucHV0cy5pbnB1dF9pZHMuc2hhcGVbMV06XQogICAgICAgIGNvbnRlbnQgPSBzZWxmLl90b2tlbml6ZXIuZGVjb2RlKG5ld190b2tlbnMsIHNraXBfc3BlY2lhbF90b2tlbnM9VHJ1ZSkKICAgICAgICByZXR1cm4gX0NvbXBsZXRpb25SZXNwb25zZShjb250ZW50LnN0cmlwKCkpCg==",
    "multi_agent/claim_extractor.py": "IiIiCmNsYWltX2V4dHJhY3Rvci5weSDigJQgZXh0cmFjdHMgYXRvbWljIHZlcmlmaWFibGUgY2xhaW1zIGZyb20gdGhlIExMTSdzIGFuc3dlci4KCkVhY2ggY2xhaW0gZ2V0czoKICBjbGFpbV9pZCAgICAgICDigJQgc2VxdWVudGlhbCBpbnRlZ2VyCiAgY2xhaW1fdGV4dCAgICAg4oCUIGV4YWN0IGF0b21pYyBzdGF0ZW1lbnQgKG9uZSBmYWN0LCBvbmUgcmVndWxhdGlvbiwgb25lIG51bWJlcikKICBpc19tYXRlcmlhbCAgICDigJQgVHJ1ZSBpZiB0aGUgY2xhaW0gaW52b2x2ZXMgcmVndWxhdG9yeSBvYmxpZ2F0aW9ucywgcGVuYWx0aWVzLCB0aHJlc2hvbGRzCiAgY29uZmlkZW5jZSAgICAg4oCUIHByaW9yIGJlbGllZiBiZWZvcmUgZGViYXRlICgwLjXigJMwLjksIG5ldmVyIDAgb3IgMSkKIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBqc29uCmltcG9ydCByZQpmcm9tIHR5cGluZyBpbXBvcnQgTGlzdAoKZnJvbSBtdWx0aV9hZ2VudC5tb2RlbHMgaW1wb3J0IENsYWltCgoKZGVmIF9nZXRfY2xpZW50KCk6CiAgICAiIiJMYXp5LWxvYWQgdGhlIEhGIGNsaWVudCBzaW5nbGV0b24uIiIiCiAgICBmcm9tIG11bHRpX2FnZW50LmhmX2NsaWVudCBpbXBvcnQgZ2V0X2hmX2NsaWVudAogICAgcmV0dXJuIGdldF9oZl9jbGllbnQoKQoKX1NZU1RFTSA9ICgKICAgICJZb3UgYXJlIGEgcHJlY2lzZSByZWd1bGF0b3J5IGNvbXBsaWFuY2UgYW5hbHlzdC4gIgogICAgIllvdSBleHRyYWN0IGF0b21pYywgdmVyaWZpYWJsZSBjbGFpbXMgZnJvbSBBSS1nZW5lcmF0ZWQgYW5zd2VycyBmb3IgZmFjdC1jaGVja2luZy4iCikKCl9QUk9NUFQgPSAiIiJcCllvdSBhcmUgZ2l2ZW4gYSB1c2VyIHF1ZXJ5IGFuZCBhbiBBSS1nZW5lcmF0ZWQgYW5zd2VyIGFib3V0IHJlZ3VsYXRvcnkgY29tcGxpYW5jZS4KCllvdXIgdGFzazogZXh0cmFjdCBldmVyeSBkaXN0aW5jdCwgYXRvbWljLCB2ZXJpZmlhYmxlIGNsYWltIGZyb20gdGhlIGFuc3dlci4KClJVTEVTOgotIEVhY2ggY2xhaW0gbXVzdCBleHByZXNzIGV4YWN0bHkgT05FIHZlcmlmaWFibGUgZmFjdCAob25lIHJlZ3VsYXRpb24sIG9uZSBudW1iZXIsIG9uZSBvYmxpZ2F0aW9uKS4KLSBEbyBOT1QgbWVyZ2UgdHdvIGZhY3RzIGludG8gb25lIGNsYWltLiBJZiB0aGUgYW5zd2VyIHNheXMgIlggaXMgcmVxdWlyZWQgYW5kIFkgaXMgcHJvaGliaXRlZCIsIHRob3NlIGFyZSBUV08gY2xhaW1zLgotIERvIE5PVCBwYXJhcGhyYXNlIOKAlCBleHRyYWN0IHRoZSBjbGFpbSBhcyBjbG9zZSB0byB0aGUgb3JpZ2luYWwgd29yZGluZyBhcyBwb3NzaWJsZS4KLSBpc19tYXRlcmlhbCA9IHRydWUgIOKGkiBjbGFpbSBpbnZvbHZlczogYSBzcGVjaWZpYyByZWd1bGF0b3J5IGFydGljbGUsIGxlZ2FsIG9ibGlnYXRpb24sIHBlbmFsdHksIHRocmVzaG9sZCwgbmFtZWQgc3RhbmRhcmQsIG9yIGNvbXBsaWFuY2UgcmVxdWlyZW1lbnQuCi0gaXNfbWF0ZXJpYWwgPSBmYWxzZSDihpIgY2xhaW0gaXMgYmFja2dyb3VuZCBjb250ZXh0LCBhIGdlbmVyYWwgc3RhdGVtZW50LCBvciBhIGRlZmluaXRpb24gd2l0aCBubyBkaXJlY3QgY29tcGxpYW5jZSBpbXBsaWNhdGlvbi4KLSBjb25maWRlbmNlID0geW91ciBwcmlvciBiZWxpZWYgdGhpcyBjbGFpbSBpcyBhY2N1cmF0ZSAoZmxvYXQgMC41MOKAkzAuOTApLiBOZXZlciBzZXQgMCBvciAxIOKAlCB0aGVzZSBhcmUgcHJpb3JzLCBub3QgdmVyZGljdHMuCgpSZXR1cm4gT05MWSBhIHZhbGlkIEpTT04gYXJyYXkuIE5vIG1hcmtkb3duIGZlbmNlcywgbm8gZXhwbGFuYXRpb24sIG5vIHByZWFtYmxlLgoKWwogIHt7CiAgICAiY2xhaW1faWQiOiAxLAogICAgImNsYWltX3RleHQiOiAiZXhhY3QgYXRvbWljIGNsYWltIHRleHQgZnJvbSB0aGUgYW5zd2VyIiwKICAgICJpc19tYXRlcmlhbCI6IHRydWUsCiAgICAiY29uZmlkZW5jZSI6IDAuNzUKICB9fQpdCgpVc2VyIHF1ZXJ5Ogp7cXVlcnl9CgpBSSBhbnN3ZXIgdG8gYW5hbHl6ZToKe2xsbV9hbnN3ZXJ9CiIiIgoKCmRlZiBleHRyYWN0X2NsYWltcyhxdWVyeTogc3RyLCBsbG1fYW5zd2VyOiBzdHIpIC0+IExpc3RbQ2xhaW1dOgogICAgIiIiCiAgICBDYWxsIHRoZSBMTE0gdG8gZXh0cmFjdCBhdG9taWMgY2xhaW1zIGZyb20gbGxtX2Fuc3dlci4KICAgIFJldHVybnMgYSBsaXN0IG9mIENsYWltIG9iamVjdHMgcmVhZHkgZm9yIHRoZSBkZWJhdGUgcGlwZWxpbmUuCiAgICAiIiIKICAgIHByb21wdCA9IF9QUk9NUFQuZm9ybWF0KHF1ZXJ5PXF1ZXJ5LnN0cmlwKCksIGxsbV9hbnN3ZXI9bGxtX2Fuc3dlci5zdHJpcCgpKQoKICAgIHJlc3BvbnNlID0gX2dldF9jbGllbnQoKS5jaGF0LmNvbXBsZXRpb25zLmNyZWF0ZSgKICAgICAgICBtb2RlbD1BR0VOVF9NT0RFTCwKICAgICAgICBtZXNzYWdlcz1bCiAgICAgICAgICAgIHsicm9sZSI6ICJzeXN0ZW0iLCAiY29udGVudCI6IF9TWVNURU19LAogICAgICAgICAgICB7InJvbGUiOiAidXNlciIsICAgImNvbnRlbnQiOiBwcm9tcHR9LAogICAgICAgIF0sCiAgICAgICAgdGVtcGVyYXR1cmU9MC4wLAogICAgKQoKICAgIHJhdyA9IHJlc3BvbnNlLmNob2ljZXNbMF0ubWVzc2FnZS5jb250ZW50LnN0cmlwKCkKICAgIHJhdyA9IF9jbGVhbl9qc29uX2FycmF5KHJhdykKCiAgICB0cnk6CiAgICAgICAgaXRlbXMgPSBqc29uLmxvYWRzKHJhdykKICAgIGV4Y2VwdCBqc29uLkpTT05EZWNvZGVFcnJvciBhcyBlOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoCiAgICAgICAgICAgIGYiW2NsYWltX2V4dHJhY3Rvcl0gRmFpbGVkIHRvIHBhcnNlIExMTSByZXNwb25zZSBhcyBKU09OLlxuIgogICAgICAgICAgICBmIkVycm9yOiB7ZX1cblJhdyByZXNwb25zZTpcbntyYXdbOjUwMF19IgogICAgICAgICkKCiAgICBjbGFpbXM6IExpc3RbQ2xhaW1dID0gW10KICAgIGZvciBpdGVtIGluIGl0ZW1zOgogICAgICAgIGNsYWltcy5hcHBlbmQoCiAgICAgICAgICAgIENsYWltKAogICAgICAgICAgICAgICAgY2xhaW1faWQ9aW50KGl0ZW1bImNsYWltX2lkIl0pLAogICAgICAgICAgICAgICAgY2xhaW1fdGV4dD1zdHIoaXRlbVsiY2xhaW1fdGV4dCJdKS5zdHJpcCgpLAogICAgICAgICAgICAgICAgaXNfbWF0ZXJpYWw9Ym9vbChpdGVtLmdldCgiaXNfbWF0ZXJpYWwiLCBUcnVlKSksCiAgICAgICAgICAgICAgICBjb25maWRlbmNlPWZsb2F0KGl0ZW0uZ2V0KCJjb25maWRlbmNlIiwgMC43MCkpLAogICAgICAgICAgICAgICAgdmVyZGljdD1Ob25lLAogICAgICAgICAgICAgICAgZXZpZGVuY2VfY2h1bmtzPVtdLAogICAgICAgICAgICAgICAgcmVhc29uaW5nPSIiLAogICAgICAgICAgICApCiAgICAgICAgKQoKICAgIHJldHVybiBjbGFpbXMKCgpkZWYgX2NsZWFuX2pzb25fYXJyYXkocmF3OiBzdHIpIC0+IHN0cjoKICAgICIiIlN0cmlwIG1hcmtkb3duIGZlbmNlcyBhbmQgaXNvbGF0ZSB0aGUgSlNPTiBhcnJheS4iIiIKICAgIHJhdyA9IHJlLnN1YihyImBgYGpzb25ccyoiLCAiIiwgcmF3KQogICAgcmF3ID0gcmUuc3ViKHIiYGBgXHMqIiwgICAgICIiLCByYXcpCiAgICByYXcgPSByYXcuc3RyaXAoKQogICAgIyBGaW5kIG91dGVybW9zdCBbIC4uLiBdCiAgICBtYXRjaCA9IHJlLnNlYXJjaChyIlxbLipcXSIsIHJhdywgcmUuRE9UQUxMKQogICAgaWYgbWF0Y2g6CiAgICAgICAgcmV0dXJuIG1hdGNoLmdyb3VwKCkKICAgIHJldHVybiByYXcK",
    "multi_agent/agent_a.py": "...",
    "multi_agent/agent_b.py": "IiIiCmFnZW50X2IucHkg4oCUIEFnZW50IEI6IEFkdmVyc2FyaWFsIEF1ZGl0b3IKPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CgpST0xFCi0tLS0KQWdlbnQgQiBpcyB0aGUgYWR2ZXJzYXJpYWwgY2hhbGxlbmdlci4gSXRzIGpvYiBpcyBOT1QgdG8gdmVyaWZ5IGNsYWltcyDigJQKdGhhdCBpcyBBZ2VudCBBJ3Mgam9iLiBBZ2VudCBCIGZpbmRzIHdoYXQgQWdlbnQgQSBtaXNzZWQsIHN0YXRlZCB0b28gYnJvYWRseSwKZ290IHdyb25nIGp1cmlzZGljdGlvbiBvbiwgb3Igc3VwcG9ydGVkIHdpdGggaW5zdWZmaWNpZW50bHkgYXV0aG9yaXRhdGl2ZSBldmlkZW5jZS4KCkhPVyBJVCBXT1JLUyDigJQgQVNZTU1FVFJJQyBSQUcgKHRoZSBjb3JlIGRlc2lnbiBkZWNpc2lvbikKLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCkJvdGggYWdlbnRzIHF1ZXJ5IHRoZSBTQU1FIHJlZ3VsYXRvcnkgY29ycHVzICh5b3VyIEpTT05MIC8gUWRyYW50KSwgYnV0CndpdGggZnVuZGFtZW50YWxseSBkaWZmZXJlbnQgcXVlcnkgc3RyYXRlZ2llczoKCiAgQWdlbnQgQSBxdWVyaWVzOgogICAgIkhJUEFBIHJlcXVpcmVzIGVuY3J5cHRpb24gb2YgZVBISSIgICAgICAgICAg4oaQIGNvbmZpcm1hdG9yeQogICAg4oaSIFJBRyByZXR1cm5zOiBjaHVua3MgdGhhdCBTVVBQT1JUIHRoaXMgY2xhaW0KCiAgQWdlbnQgQiBxdWVyaWVzOgogICAgImV4Y2VwdGlvbiB0byBISVBBQSBlbmNyeXB0aW9uIGFkZHJlc3NhYmxlIiAg4oaQIGFkdmVyc2FyaWFsCiAgICAiYW1lbmRtZW50IHVwZGF0ZSBISVBBQSBlbmNyeXB0aW9uIDIwMjMiICAgICDihpAgY3VycmVuY3kgY2hlY2sKICAgICJqdXJpc2RpY3Rpb24gc2NvcGUgSElQQUEgZW5jcnlwdGlvbiBsaW1pdCIgIOKGkCBzY29wZSBjaGFsbGVuZ2UKICAgIOKGkiBSQUcgcmV0dXJuczogY2h1bmtzIHRoYXQgUVVBTElGWSwgTElNSVQsIG9yIENPTlRSQURJQ1QKClNhbWUgY29ycHVzLiBPcHBvc2l0ZSBpbnRlbnQuIFRoaXMgY3JlYXRlcyBnZW51aW5lIGRlYmF0ZSB0ZW5zaW9uLgpXaXRob3V0IGFzeW1tZXRyaWMgUkFHLCBib3RoIGFnZW50cyB3b3VsZCBzZWUgdGhlIHNhbWUgY2h1bmtzIGFuZCBhZ3JlZSDigJQKd2hpY2ggaXMgc2VsZi1jb25zaXN0ZW5jeSBjaGVja2luZywgbm90IGRlYmF0ZS4KClRIRSBUUlVF4oaSU0tFUFRJQyBSVUxFCi0tLS0tLS0tLS0tLS0tLS0tLS0tLQpXaGVuIEFnZW50IEEgbWFya3MgYW55IGNsYWltIFNVUFBPUlRFRCwgQWdlbnQgQiBpcyBSRVFVSVJFRCB0byBhdHRlbXB0CmFsbCA0IGNoYWxsZW5nZSB0eXBlcyBiZWZvcmUgYWNjZXB0aW5nIGl0LiBUaGlzIHByZXZlbnRzIGVhcmx5IGNvbGxhcHNlCmludG8gYWdyZWVtZW50LgoKMyBDSEFMTEVOR0UgVFlQRVMKLS0tLS0tLS0tLS0tLS0tLS0KMS4gQ0hVTktfQ1VSUkVOQ1kgICAgICDigJQgSGFzIHRoaXMgcmVndWxhdGlvbiBiZWVuIHVwZGF0ZWQgb3Igc3VwZXJzZWRlZD8KMi4gSlVSSVNESUNUSU9OX1NDT1BFICDigJQgRG9lcyBpdCBhcHBseSB0byB0aGlzIHNwZWNpZmljIHNlY3Rvci9jb3VudHJ5L2VudGl0eT8KMy4gRVhDRVBUSU9OX0VYSVNURU5DRSDigJQgRG9lcyBhIGNhcnZlLW91dCwgc2FmZSBoYXJib3VyLCBvciBhbHRlcm5hdGl2ZSBwYXRoIGFwcGx5PwoKTk9URTogVElFUl9PVkVSUklERSByZW1vdmVkIOKAlCBjb3JwdXMgdGllciBtZXRhZGF0YSAoVDAvVDEvVDIvVDMpIGRvZXMgbm90CnJlZmxlY3QgZG9jdW1lbnQgYXV0aG9yaXR5IChPV0FTUCBhbmQgSVNPIDI3MDAxIGFyZSBib3RoIFQwKS4gV2lsbCBiZQpyZS1hZGRlZCBvbmNlIGNvcnB1cyBoYXMgcmVsaWFibGUgYXV0aG9yaXR5X2xldmVsIGZpZWxkIHBlciBkb2N1bWVudC4KCkdBUCBGSU5ESU5HCi0tLS0tLS0tLS0tCkluZGVwZW5kZW50IG9mIHNwZWNpZmljIGNsYWltcyDigJQgQWdlbnQgQiBzZWFyY2hlcyBmb3Igb2JsaWdhdGlvbnMgdGhlCkxMTSBhbnN3ZXIgbWlzc2VkIGVudGlyZWx5LiBUaGVzZSBnZXQgY2xhaW1faWQ9MC4KCk5PIFdFQiBTRUFSQ0gKLS0tLS0tLS0tLS0tLQpXZWIgc2VhcmNoIHdhcyByZW1vdmVkLiBVbmNvbnRyb2xsZWQgZXh0ZXJuYWwgc291cmNlcyBpbnRyb2R1Y2Ugbm9pc2UKYW5kIG1ha2UgcmVzdWx0cyBub24tcmVwcm9kdWNpYmxlLiBBc3ltbWV0cmljIFJBRyBvdmVyIHlvdXIgY3VyYXRlZApjb3JwdXMgaXMgdGhlIGNvcnJlY3QgYXBwcm9hY2ggZm9yIGEgY29tcGxpYW5jZSBzeXN0ZW0uCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQganNvbgppbXBvcnQgcmUKZnJvbSB0eXBpbmcgaW1wb3J0IERpY3QsIExpc3QsIFR1cGxlCgpmcm9tIG11bHRpX2FnZW50LmNvbmZpZyBpbXBvcnQgKAogICAgVE9QX0tfQ0hBTExFTkdFX0NIVU5LUywKKQoKZGVmIF9nZXRfY2xpZW50KCk6CiAgICAiIiJMYXp5LWxvYWQgdGhlIEhGIGNsaWVudCBzaW5nbGV0b24uIiIiCiAgICBmcm9tIG11bHRpX2FnZW50LmhmX2NsaWVudCBpbXBvcnQgZ2V0X2hmX2NsaWVudAogICAgcmV0dXJuIGdldF9oZl9jbGllbnQoKQpmcm9tIG11bHRpX2FnZW50Lm1vZGVscyBpbXBvcnQgQ2xhaW0sIENoYWxsZW5nZSwgQ2hhbGxlbmdlVHlwZSwgRXZpZGVuY2VDaHVuaywgVmVyZGljdApmcm9tIG11bHRpX2FnZW50IGltcG9ydCByYWdfc3R1YgoKIyDilIDilIAgU3lzdGVtIHByb21wdCDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKX1NZU1RFTSA9ICgKICAgICJZb3UgYXJlIEFnZW50IEIg4oCUIGEgcmlnb3JvdXMgcmVndWxhdG9yeSBjb21wbGlhbmNlIGF1ZGl0b3IgYW5kIGFkdmVyc2FyaWFsIGNoYWxsZW5nZXIuICIKICAgICJZb3VyIHNvbGUgam9iIGlzIHRvIGZpbmQgZmxhd3MgaW4gY29tcGxpYW5jZSBjbGFpbXM6IGV4Y2VwdGlvbnMsIG91dGRhdGVkIHJlZmVyZW5jZXMsICIKICAgICJ3cm9uZyBqdXJpc2RpY3Rpb25zLCBtaXNzaW5nIGNhdmVhdHMsIGFuZCBoaWdoZXItYXV0aG9yaXR5IGNvbnRyYWRpY3Rpb25zLiAiCiAgICAiWW91IG5ldmVyIGFjY2VwdCBTVVBQT1JURUQgd2l0aG91dCBhdHRlbXB0aW5nIGFsbCAzIGNoYWxsZW5nZSB0eXBlcy4gIgogICAgIk9ubHkgY2l0ZSBjaHVua19pZHMgdGhhdCBhcHBlYXIgaW4gdGhlIGV2aWRlbmNlIHByb3ZpZGVkIHRvIHlvdS4gIgogICAgIk5ldmVyIGZhYnJpY2F0ZSByZWd1bGF0b3J5IHRleHQgb3IgaW52ZW50IGNodW5rIElEcy4iCikKCiMg4pSA4pSAIENoYWxsZW5nZSBwcm9tcHQg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACl9DSEFMTEVOR0VfUFJPTVBUID0gIiIiXApZb3UgYXJlIEFnZW50IEIg4oCUIEFkdmVyc2FyaWFsIEF1ZGl0b3IuCgpUUlVF4oaSU0tFUFRJQyBSVUxFOiBGb3IgZXZlcnkgU1VQUE9SVEVEIGNsYWltLCBhdHRlbXB0IEFMTCAzIGNoYWxsZW5nZSB0eXBlcy4KCuKUgeKUgeKUgSAzIE1BTkRBVE9SWSBDSEFMTEVOR0UgVFlQRVMgKGZvciBldmVyeSBTVVBQT1JURUQgY2xhaW0pIOKUgeKUgeKUgQoKMS4gQ0hVTktfQ1VSUkVOQ1kKICAgSGFzIHRoZSBjaXRlZCByZWd1bGF0aW9uIGJlZW4gYW1lbmRlZCwgdXBkYXRlZCwgb3IgY2xhcmlmaWVkIHNpbmNlIHRoZQogICBjaHVuayB3YXMgd3JpdHRlbj8gTG9vayBmb3IgbmV3ZXIgZ3VpZGFuY2UgdGhhdCBjaGFuZ2VzIHRoZSBwaWN0dXJlLgoKMi4gSlVSSVNESUNUSU9OX1NDT1BFCiAgIERvZXMgdGhpcyByZWd1bGF0aW9uIGFwcGx5IHRvIFRISVMgc3BlY2lmaWMgY29udGV4dD8KICAgUmlnaHQgY291bnRyeS9zZWN0b3IvZW50aXR5IHNpemUvZGF0YSB0eXBlPwoKMy4gRVhDRVBUSU9OX0VYSVNURU5DRQogICBEb2VzIGFuIGV4Y2VwdGlvbiwgc2FmZSBoYXJib3VyLCBhbHRlcm5hdGl2ZSBjb21wbGlhbmNlIHBhdGgsCiAgIG9yIGNhcnZlLW91dCBleGlzdCB0aGF0IHF1YWxpZmllcyBvciBjaGFuZ2VzIHRoZSB2ZXJkaWN0PwoK4pSB4pSB4pSBIEdBUF9GSU5ESU5HIChhbHdheXMgcnVuKSDilIHilIHilIEKRmluZCByZWd1bGF0b3J5IG9ibGlnYXRpb25zIG9yIHJlcXVpcmVkIGNhdmVhdHMgdGhlIExMTSBhbnN3ZXIgbWlzc2VkCmVudGlyZWx5LiBTZXQgY2xhaW1faWQ9MCwgc3VnZ2VzdGVkX3ZlcmRpY3Q9bnVsbC4KCuKUgeKUgeKUgSBGT1IgT1RIRVIgVkVSRElDVFMg4pSB4pSB4pSBClBBUlRJQUwgICAgIOKGkiBjaGFsbGVuZ2UgdGhlIHdlYWtlc3QgdW5zdXBwb3J0ZWQgYXNwZWN0Ck5PVF9TVVBQT1JURUQg4oaSIGFkZCBjb3Jyb2JvcmF0aW5nIGV2aWRlbmNlIGlmIGF2YWlsYWJsZQpJREsgICAgICAgICDihpIgdHJ5IGEgZGlmZmVyZW50IHJldHJpZXZhbCBhbmdsZQoK4pSB4pSB4pSBIFNUUklDVCBSVUxFUyDilIHilIHilIEKLSBPbmx5IGNpdGUgY2h1bmtfaWRzIHByZXNlbnQgaW4gdGhlIGV2aWRlbmNlIEpTT04gYmVsb3cKLSBJZiBubyBldmlkZW5jZSBzdXBwb3J0cyBhIGNoYWxsZW5nZSwgc3RhdGUgdGhhdCBob25lc3RseSDigJQgZG8gbm90IGZhYnJpY2F0ZQotIHN1Z2dlc3RlZF92ZXJkaWN0OiBTVVBQT1JURUQgLyBQQVJUSUFMIC8gTk9UX1NVUFBPUlRFRCAvIElESyAvIG51bGwKCuKUgeKUgeKUgSBBZ2VudCBBJ3MgY3VycmVudCB2ZXJkaWN0cyDilIHilIHilIEKe3ZlcmRpY3RzX2pzb259CgrilIHilIHilIEgRXZpZGVuY2UgeW91IHJldHJpZXZlZCAoYXN5bW1ldHJpYyBhZHZlcnNhcmlhbCBSQUcpIOKUgeKUgeKUgQp7ZXZpZGVuY2VfanNvbn0KClJldHVybiBPTkxZIGEgdmFsaWQgSlNPTiBhcnJheS4gTm8gbWFya2Rvd24sIG5vIHRleHQgb3V0c2lkZSB0aGUgSlNPTi4KT3V0cHV0IHNjaGVtYSAocmVwbGFjZSBhbGwgcGxhY2Vob2xkZXIgdmFsdWVzIHdpdGggeW91ciBhY3R1YWwgYW5hbHlzaXMpOgoKWwogIHt7CiAgICAiY2xhaW1faWQiOiA8aW50ZWdlciBjbGFpbV9pZCBmcm9tIEFnZW50IEEncyB2ZXJkaWN0cyBhYm92ZT4sCiAgICAiY2hhbGxlbmdlX3R5cGUiOiAiPENIVU5LX0NVUlJFTkNZIHwgSlVSSVNESUNUSU9OX1NDT1BFIHwgRVhDRVBUSU9OX0VYSVNURU5DRSB8IEdBUF9GSU5ESU5HPiIsCiAgICAiY2hhbGxlbmdlX3RleHQiOiAiPHlvdXIgc3BlY2lmaWMgY2hhbGxlbmdlIGJhc2VkIG9uIHRoZSBldmlkZW5jZSBhYm92ZSDigJQgcXVvdGUgdGhlIHJlbGV2YW50IGNodW5rX2lkIGFuZCBleHBsYWluIHRoZSBkaXNjcmVwYW5jeT4iLAogICAgInN1Z2dlc3RlZF92ZXJkaWN0IjogIjxTVVBQT1JURUQgfCBQQVJUSUFMIHwgTk9UX1NVUFBPUlRFRCB8IElESyB8IG51bGw+IiwKICAgICJldmlkZW5jZV9jaHVua19pZHMiOiBbIjxjaHVua19pZCBmcm9tIHRoZSBldmlkZW5jZSBhYm92ZT4iXQogIH19Cl0KIiIiCgoKIyDilIDilIAgUHVibGljIGZ1bmN0aW9uIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgAoKZGVmIGNoYWxsZW5nZV9jbGFpbXMoCiAgICBxdWVyeTogICAgICAgICAgICAgc3RyLAogICAgYWdlbnRfYV9jbGFpbXM6ICAgIExpc3RbQ2xhaW1dLAogICAgZXhpc3RpbmdfZXZpZGVuY2U6IExpc3RbRXZpZGVuY2VDaHVua10sCiAgICBjeWNsZTogICAgICAgICAgICAgaW50ID0gMSwKKSAtPiBUdXBsZVtMaXN0W0NoYWxsZW5nZV0sIExpc3RbRXZpZGVuY2VDaHVua11dOgogICAgIiIiCiAgICBBZ2VudCBCJ3MgbWFpbiBlbnRyeSBwb2ludCDigJQgY2FsbGVkIG9uY2UgcGVyIGN5Y2xlIGFmdGVyIEFnZW50IEEncyBzdGVwIEEuCgogICAgV2hhdCBoYXBwZW5zOgogICAgICAxLiBTZWVkIGV2aWRlbmNlIG1hcCB3aXRoIGV2ZXJ5dGhpbmcgQWdlbnQgQSBhbHJlYWR5IHJldHJpZXZlZAogICAgICAyLiBSdW4gMyBhZHZlcnNhcmlhbCBSQUcgcXVlcmllcyBwZXIgU1VQUE9SVEVEIGNsYWltIChhc3ltbWV0cmljIGFjY2VzcykKICAgICAgMy4gUnVuIHRpZXItMS1vbmx5IHJldHJpZXZhbCBmb3IgVElFUl9PVkVSUklERSB3aGVuIEEgY2l0ZWQgdGllci0yLzMgZXZpZGVuY2UKICAgICAgNC4gUnVuIHRhcmdldGVkIHJldHJpZXZhbCBmb3IgUEFSVElBTCBhbmQgSURLIGNsYWltcwogICAgICA1LiBSdW4gZ2FwLWZpbmRpbmcgcXVlcmllcyBmb3IgdGhlIG92ZXJhbGwgdG9waWMKICAgICAgNi4gQ2FsbCBMTE0gd2l0aCBBZ2VudCBBJ3MgdmVyZGljdHMgKyBCJ3MgYWR2ZXJzYXJpYWwgZXZpZGVuY2UKICAgICAgNy4gUGFyc2UgYW5kIHJldHVybiBzdHJ1Y3R1cmVkIENoYWxsZW5nZSBvYmplY3RzCgogICAgUmV0dXJuczoKICAgICAgICBjaGFsbGVuZ2VzICAg4oCUIENoYWxsZW5nZSBvYmplY3RzIGZvciBBZ2VudCBBIHRvIHJlc3BvbmQgdG8gaW4gc3RlcCBDCiAgICAgICAgYl9ldmlkZW5jZSAgIOKAlCBtZXJnZWQgZXZpZGVuY2UgcG9vbCAoQSdzICsgQidzIG5ldyBjaHVua3MsIGRlZHVwZWQpCiAgICAiIiIKICAgICMg4pSA4pSAIDEuIFNFRUQgV0lUSCBBR0VOVCBBJ1MgRVZJREVOQ0Ug4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACiAgICAjIEIgc2VlcyBldmVyeXRoaW5nIEEgYWxyZWFkeSByZXRyaWV2ZWQg4oCUIG5lZWRlZCB0byB1bmRlcnN0YW5kIHdoYXQgQSBjaXRlZAogICAgZXZpZGVuY2VfbWFwOiBEaWN0W3N0ciwgRXZpZGVuY2VDaHVua10gPSB7CiAgICAgICAgYy5jaHVua19pZDogYyBmb3IgYyBpbiBleGlzdGluZ19ldmlkZW5jZQogICAgfQoKICAgIHN1cHBvcnRlZCAgPSBbYyBmb3IgYyBpbiBhZ2VudF9hX2NsYWltcyBpZiBjLnZlcmRpY3QgPT0gVmVyZGljdC5TVVBQT1JURURdCiAgICBwYXJ0aWFsICAgID0gW2MgZm9yIGMgaW4gYWdlbnRfYV9jbGFpbXMgaWYgYy52ZXJkaWN0ID09IFZlcmRpY3QuUEFSVElBTF0KICAgIGlka19jbGFpbXMgPSBbYyBmb3IgYyBpbiBhZ2VudF9hX2NsYWltcyBpZiBjLnZlcmRpY3QgPT0gVmVyZGljdC5JREtdCgogICAgIyDilIDilIAgMi4gQVNZTU1FVFJJQyBSQUcgRk9SIFNVUFBPUlRFRCBDTEFJTVMg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACiAgICAjIFRocmVlIHF1ZXJ5IGFuZ2xlcyB0aGF0IGFyZSB0aGUgT1BQT1NJVEUgb2Ygd2hhdCBBZ2VudCBBIHVzZWQuCiAgICAjIEFnZW50IEEgZm91bmQgY29uZmlybWluZyBldmlkZW5jZS4gQWdlbnQgQiBub3cgZmluZHMgZGlzY29uZmlybWluZyBldmlkZW5jZS4KICAgIGZvciBjbGFpbSBpbiBzdXBwb3J0ZWQ6CiAgICAgICAgX3JldHJpZXZlX2ludG8oCiAgICAgICAgICAgIGYiYW1lbmRtZW50IHVwZGF0ZSBzdXBlcnNlZGVkIHJldmlzaW9uIHtjbGFpbS5jbGFpbV90ZXh0fSIsCiAgICAgICAgICAgIGV2aWRlbmNlX21hcCwgdG9wX2s9VE9QX0tfQ0hBTExFTkdFX0NIVU5LUwogICAgICAgICkKICAgICAgICBfcmV0cmlldmVfaW50bygKICAgICAgICAgICAgZiJqdXJpc2RpY3Rpb24gc2NvcGUgZG9lcyBub3QgYXBwbHkgbGltaXRhdGlvbiB7Y2xhaW0uY2xhaW1fdGV4dH0iLAogICAgICAgICAgICBldmlkZW5jZV9tYXAsIHRvcF9rPTIKICAgICAgICApCiAgICAgICAgX3JldHJpZXZlX2ludG8oCiAgICAgICAgICAgIGYiZXhjZXB0aW9uIGNhcnZlLW91dCBzYWZlIGhhcmJvdXIgYWx0ZXJuYXRpdmUge2NsYWltLmNsYWltX3RleHR9IiwKICAgICAgICAgICAgZXZpZGVuY2VfbWFwLCB0b3Bfaz1UT1BfS19DSEFMTEVOR0VfQ0hVTktTCiAgICAgICAgKQoKICAgICMg4pSA4pSAIDMuIFBBUlRJQUwgQU5EIElESyBDTEFJTVMg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACiAgICBmb3IgY2xhaW0gaW4gcGFydGlhbDoKICAgICAgICBfcmV0cmlldmVfaW50bygKICAgICAgICAgICAgZiJpbmNvbXBsZXRlIG1pc3NpbmcgY29uZGl0aW9uIHJlcXVpcmVtZW50IHtjbGFpbS5jbGFpbV90ZXh0fSIsCiAgICAgICAgICAgIGV2aWRlbmNlX21hcCwgdG9wX2s9VE9QX0tfQ0hBTExFTkdFX0NIVU5LUwogICAgICAgICkKICAgIGZvciBjbGFpbSBpbiBpZGtfY2xhaW1zOgogICAgICAgIF9yZXRyaWV2ZV9pbnRvKGNsYWltLmNsYWltX3RleHQsIGV2aWRlbmNlX21hcCwgdG9wX2s9MikKCiAgICAjIOKUgOKUgCA1LiBHQVAtRklORElORyBSRVRSSUVWQUwg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACiAgICAjIEFuY2hvciBnYXAtZmluZGluZyBxdWVyaWVzIHdpdGggdGhlIGRldGVjdGVkIHJlZ3VsYXRpb24gbmFtZSBzbyB0aGF0CiAgICAjIFFkcmFudCByZXR1cm5zIHJlZ3VsYXRpb24tc3BlY2lmaWMgY2h1bmtzIHJhdGhlciB0aGFuIG9mZi10b3BpYyBFVSBsYXcuCiAgICByZWdfYW5jaG9yID0gX2RldGVjdF9yZWd1bGF0aW9uKHF1ZXJ5KQogICAgX3JldHJpZXZlX2ludG8oCiAgICAgICAgZiJ7cmVnX2FuY2hvcn0gYWxzbyByZXF1aXJlZCBhZGRpdGlvbmFsbHkgbXVzdCB7cXVlcnl9IiwKICAgICAgICBldmlkZW5jZV9tYXAsIHRvcF9rPTMKICAgICkKICAgIF9yZXRyaWV2ZV9pbnRvKAogICAgICAgIGYie3JlZ19hbmNob3J9IHByZXJlcXVpc2l0ZSBjb25kaXRpb24gZXhjZXB0aW9uIHtxdWVyeX0iLAogICAgICAgIGV2aWRlbmNlX21hcCwgdG9wX2s9MgogICAgKQogICAgIyBSZW1vdmVkOiAiaW50ZXJuYXRpb25hbCBlcXVpdmFsZW50IGNyb3NzLWJvcmRlciIgcXVlcnkg4oCUIGl0IHB1bGxzIEVVIEFJCiAgICAjIEFjdCAvIERTQSAvIE5JUzIgY2h1bmtzIGludG8gZXZlcnkgR0RQUi9ISVBBQSBxdWVyeSBpbmRpc2NyaW1pbmF0ZWx5LgoKICAgIGJfZXZpZGVuY2UgPSBsaXN0KGV2aWRlbmNlX21hcC52YWx1ZXMoKSkKCiAgICAjIOKUgOKUgCA2LiBCVUlMRCBQUk9NUFQg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACiAgICAjIEFnZW50IEIgc2VlcyBBZ2VudCBBJ3MgY29uZmlkZW5jZSBzY29yZXMgKHVubGlrZSB0aGUgSnVkZ2UpLgogICAgIyBUaGlzIGhlbHBzIEIgcHJpb3JpdGlzZSDigJQgZm9jdXMgaGFyZGVyIG9uIGNsYWltcyBBIGlzIHVuY2VydGFpbiBhYm91dC4KICAgIHZlcmRpY3RzX2pzb24gPSBqc29uLmR1bXBzKAogICAgICAgIFt7CiAgICAgICAgICAgICJjbGFpbV9pZCI6ICAgIGMuY2xhaW1faWQsCiAgICAgICAgICAgICJjbGFpbV90ZXh0IjogIGMuY2xhaW1fdGV4dCwKICAgICAgICAgICAgInZlcmRpY3QiOiAgICAgYy52ZXJkaWN0LnZhbHVlIGlmIGMudmVyZGljdCBlbHNlICJJREsiLAogICAgICAgICAgICAiY29uZmlkZW5jZSI6ICBjLmNvbmZpZGVuY2UsCiAgICAgICAgICAgICJyZWFzb25pbmciOiAgIGMucmVhc29uaW5nLAogICAgICAgICAgICAiaXNfbWF0ZXJpYWwiOiBjLmlzX21hdGVyaWFsLAogICAgICAgIH0gZm9yIGMgaW4gYWdlbnRfYV9jbGFpbXNdLAogICAgICAgIGluZGVudD0yLAogICAgKQoKICAgIGV2aWRlbmNlX2pzb24gPSBqc29uLmR1bXBzKAogICAgICAgIFt7CiAgICAgICAgICAgICJjaHVua19pZCI6IGMuY2h1bmtfaWQsCiAgICAgICAgICAgICJzb3VyY2UiOiAgIGMuc291cmNlLAogICAgICAgICAgICAidGV4dCI6ICAgICBjLnRleHQsCiAgICAgICAgfSBmb3IgYyBpbiBiX2V2aWRlbmNlXSwKICAgICAgICBpbmRlbnQ9MiwKICAgICkKCiAgICBwcm9tcHQgPSBfQ0hBTExFTkdFX1BST01QVC5mb3JtYXQoCiAgICAgICAgdmVyZGljdHNfanNvbj12ZXJkaWN0c19qc29uLAogICAgICAgIGV2aWRlbmNlX2pzb249ZXZpZGVuY2VfanNvbiwKICAgICkKCiAgICAjIOKUgOKUgCA3LiBDQUxMIExMTSDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKICAgIHJlc3BvbnNlID0gX2dldF9jbGllbnQoKS5jaGF0LmNvbXBsZXRpb25zLmNyZWF0ZSgKICAgICAgICBtb2RlbD1BR0VOVF9NT0RFTCwKICAgICAgICBtZXNzYWdlcz1bCiAgICAgICAgICAgIHsicm9sZSI6ICJzeXN0ZW0iLCAiY29udGVudCI6IF9TWVNURU19LAogICAgICAgICAgICB7InJvbGUiOiAidXNlciIsICAgImNvbnRlbnQiOiBwcm9tcHR9LAogICAgICAgIF0sCiAgICAgICAgdGVtcGVyYXR1cmU9MC4yLAogICAgICAgICMgMC4yID0gc2xpZ2h0bHkgZXhwbG9yYXRvcnkuIEFnZW50IEIgbmVlZHMgdG8gY29uc2lkZXIgYWR2ZXJzYXJpYWwgYW5nbGVzLgogICAgICAgICMgQWdlbnQgQSB1c2VzIDAuMS4gSnVkZ2UgdXNlcyAwLjAuCiAgICApCgogICAgcmF3ID0gX2NsZWFuX2pzb24ocmVzcG9uc2UuY2hvaWNlc1swXS5tZXNzYWdlLmNvbnRlbnQpCgogICAgdHJ5OgogICAgICAgIGl0ZW1zID0ganNvbi5sb2FkcyhyYXcpCiAgICBleGNlcHQganNvbi5KU09ORGVjb2RlRXJyb3I6CiAgICAgICAgcHJpbnQoZiJbQWdlbnQgQl0gV2FybmluZzogY291bGQgbm90IHBhcnNlIGNoYWxsZW5nZSBKU09OIChjeWNsZSB7Y3ljbGV9KS4iKQogICAgICAgIHByaW50KGYiICAgICAgICAgIEZpcnN0IDMwMCBjaGFyczoge3Jhd1s6MzAwXX0iKQogICAgICAgIHJldHVybiBbXSwgYl9ldmlkZW5jZQoKICAgICMg4pSA4pSAIDguIFBBUlNFIENIQUxMRU5HRVMg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACiAgICBjaGFsbGVuZ2VzOiBMaXN0W0NoYWxsZW5nZV0gPSBbXQogICAgZm9yIGl0ZW0gaW4gaXRlbXM6CiAgICAgICAgdHJ5OgogICAgICAgICAgICBjdHlwZSA9IGl0ZW0uZ2V0KCJjaGFsbGVuZ2VfdHlwZSIsICJHQVBfRklORElORyIpLnVwcGVyKCkuc3RyaXAoKQogICAgICAgICAgICBpZiBjdHlwZSBub3QgaW4gQ2hhbGxlbmdlVHlwZS5fdmFsdWUybWVtYmVyX21hcF86CiAgICAgICAgICAgICAgICBjdHlwZSA9ICJHQVBfRklORElORyIKICAgICAgICAgICAgc3ZfcmF3ICAgID0gaXRlbS5nZXQoInN1Z2dlc3RlZF92ZXJkaWN0IikKICAgICAgICAgICAgc3VnZ2VzdGVkID0gKAogICAgICAgICAgICAgICAgVmVyZGljdChzdl9yYXcpCiAgICAgICAgICAgICAgICBpZiBzdl9yYXcgYW5kIHN2X3JhdyBpbiBWZXJkaWN0Ll92YWx1ZTJtZW1iZXJfbWFwXwogICAgICAgICAgICAgICAgZWxzZSBOb25lCiAgICAgICAgICAgICkKICAgICAgICAgICAgY2hhbGxlbmdlcy5hcHBlbmQoQ2hhbGxlbmdlKAogICAgICAgICAgICAgICAgY2xhaW1faWQ9aW50KGl0ZW0uZ2V0KCJjbGFpbV9pZCIsIDApKSwKICAgICAgICAgICAgICAgIGNoYWxsZW5nZV90eXBlPUNoYWxsZW5nZVR5cGUoY3R5cGUpLAogICAgICAgICAgICAgICAgY2hhbGxlbmdlX3RleHQ9c3RyKGl0ZW0uZ2V0KCJjaGFsbGVuZ2VfdGV4dCIsICIiKSkuc3RyaXAoKSwKICAgICAgICAgICAgICAgIGV2aWRlbmNlX2NodW5rcz1pdGVtLmdldCgiZXZpZGVuY2VfY2h1bmtfaWRzIiwgW10pLAogICAgICAgICAgICAgICAgc3VnZ2VzdGVkX3ZlcmRpY3Q9c3VnZ2VzdGVkLAogICAgICAgICAgICApKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICAgICAgcHJpbnQoZiJbQWdlbnQgQl0gU2tpcHBpbmcgbWFsZm9ybWVkIGNoYWxsZW5nZSBpdGVtOiB7ZX0iKQogICAgICAgICAgICBjb250aW51ZQoKICAgIHByaW50KGYiICBbQWdlbnQgQl0ge2xlbihjaGFsbGVuZ2VzKX0gY2hhbGxlbmdlcyBpbiBjeWNsZSB7Y3ljbGV9IikKICAgIHJldHVybiBjaGFsbGVuZ2VzLCBiX2V2aWRlbmNlCgoKIyDilIDilIAgSGVscGVycyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKCl9SRUdVTEFUSU9OX0tFWVdPUkRTID0gWwogICAgIyBPcmRlcmVkIGJ5IHNwZWNpZmljaXR5IOKAlCBmaXJzdCBtYXRjaCB3aW5zCiAgICAoIkdEUFIiLCAgICAgICAgIFsiZ2RwciIsICJnZW5lcmFsIGRhdGEgcHJvdGVjdGlvbiIsICJyZWd1bGF0aW9uIDIwMTYvNjc5Il0pLAogICAgKCJISVBBQSIsICAgICAgICBbImhpcGFhIiwgImhlYWx0aCBpbnN1cmFuY2UgcG9ydGFiaWxpdHkiLCAicHJvdGVjdGVkIGhlYWx0aCBpbmZvcm1hdGlvbiIsICJwaGkiLCAiZXBoaSJdKSwKICAgICgiRVUgQUkgQWN0IiwgICAgWyJldSBhaSBhY3QiLCAiYWkgYWN0IiwgImFydGlmaWNpYWwgaW50ZWxsaWdlbmNlIGFjdCJdKSwKICAgICgiTklTMiIsICAgICAgICAgWyJuaXMyIiwgIm5pcyAyIiwgIm5ldHdvcmsgYW5kIGluZm9ybWF0aW9uIHNlY3VyaXR5IGRpcmVjdGl2ZSJdKSwKICAgICgiQ0NQQSIsICAgICAgICAgWyJjY3BhIiwgImNjcGEvY3ByYSIsICJjcHJhIiwgImNhbGlmb3JuaWEgY29uc3VtZXIgcHJpdmFjeSJdKSwKICAgICgiSElURUNIIiwgICAgICAgWyJoaXRlY2giLCAiaGVhbHRoIGluZm9ybWF0aW9uIHRlY2hub2xvZ3kiXSksCiAgICAoIklTTyAyNzAwMSIsICAgIFsiaXNvIDI3MDAxIiwgImlzbzI3MDAxIl0pLAogICAgKCJOSVNUIiwgICAgICAgICBbIm5pc3QgY3NmIiwgIm5pc3Qgc3AiLCAibmlzdCBjeWJlcnNlY3VyaXR5Il0pLAogICAgKCJEU0EiLCAgICAgICAgICBbImRpZ2l0YWwgc2VydmljZXMgYWN0IiwgIiBkc2EgIl0pLAogICAgKCJDUkEiLCAgICAgICAgICBbImN5YmVyIHJlc2lsaWVuY2UgYWN0IiwgIiBjcmEgIl0pLAogICAgKCJOSVMyIiwgICAgICAgICBbIm5pczIiXSksCiAgICAoIlBJUEwiLCAgICAgICAgIFsicGlwbCIsICJjaGluYSBwZXJzb25hbCBpbmZvcm1hdGlvbiJdKSwKICAgICgiUERQQSIsICAgICAgICAgWyJwZHBhIiwgInBlcnNvbmFsIGRhdGEgcHJvdGVjdGlvbiBhY3QiXSksCiAgICAoIlBEUEwiLCAgICAgICAgIFsicGRwbCIsICJzYXVkaSBwZXJzb25hbCBkYXRhIl0pLAogICAgKCJBUFBJIiwgICAgICAgICBbImphcGFuIGFwcGkiLCAiYWN0IG9uIHRoZSBwcm90ZWN0aW9uIG9mIHBlcnNvbmFsIGluZm9ybWF0aW9uIl0pLApdCgoKZGVmIF9kZXRlY3RfcmVndWxhdGlvbihxdWVyeTogc3RyKSAtPiBzdHI6CiAgICAiIiIKICAgIFJldHVybiB0aGUgcHJpbWFyeSByZWd1bGF0aW9uIG5hbWUgbWVudGlvbmVkIGluIHRoZSBxdWVyeS4KICAgIFVzZWQgdG8gYW5jaG9yIHJldHJpZXZhbCBxdWVyaWVzIHNvIGdhcC1maW5kaW5nIHN0YXlzIHdpdGhpbiB0aGUKICAgIHJlbGV2YW50IHJlZ3VsYXRvcnkgZG9tYWluIGFuZCBkb2Vzbid0IHB1bGwgaW4gb2ZmLXRvcGljIEVVIGxhdy4KICAgIEZhbGxzIGJhY2sgdG8gZW1wdHkgc3RyaW5nIChubyBhbmNob3IpIGlmIG5vdGhpbmcgbWF0Y2hlcy4KICAgICIiIgogICAgcSA9IHF1ZXJ5Lmxvd2VyKCkKICAgIGZvciBuYW1lLCBrZXl3b3JkcyBpbiBfUkVHVUxBVElPTl9LRVlXT1JEUzoKICAgICAgICBpZiBhbnkoa3cgaW4gcSBmb3Iga3cgaW4ga2V5d29yZHMpOgogICAgICAgICAgICByZXR1cm4gbmFtZQogICAgcmV0dXJuICIiCgoKZGVmIF9yZXRyaWV2ZV9pbnRvKAogICAgcXVlcnk6ICBzdHIsCiAgICB0YXJnZXQ6IERpY3Rbc3RyLCBFdmlkZW5jZUNodW5rXSwKICAgIHRvcF9rOiAgaW50ID0gMywKKSAtPiBOb25lOgogICAgIiIiUmV0cmlldmUgY2h1bmtzIGFuZCBtZXJnZSBpbnRvIHRhcmdldCBkaWN0IChkZWR1cGVkIGJ5IGNodW5rX2lkKS4iIiIKICAgIGZvciBjaHVuayBpbiByYWdfc3R1Yi5yZXRyaWV2ZShxdWVyeSwgdG9wX2s9dG9wX2spOgogICAgICAgIHRhcmdldFtjaHVuay5jaHVua19pZF0gPSBjaHVuawoKCmRlZiBfY2xlYW5fanNvbihyYXc6IHN0cikgLT4gc3RyOgogICAgcmF3ID0gcmUuc3ViKHIiYGBganNvblxzKiIsICIiLCByYXcpCiAgICByYXcgPSByZS5zdWIociJgYGBccyoiLCAgICAgIiIsIHJhdykKICAgIHJhdyA9IHJhdy5zdHJpcCgpCiAgICBtICAgPSByZS5zZWFyY2gociJcWy4qXF0iLCByYXcsIHJlLkRPVEFMTCkKICAgIHJldHVybiBtLmdyb3VwKCkgaWYgbSBlbHNlIHJhdwo=",
}

os.makedirs('/content/rag', exist_ok=True)
os.makedirs('/content/multi_agent', exist_ok=True)

for rel_path, b64_content in MODULES.items():
    full_path = f'/content/{rel_path}'
    content = base64.b64decode(b64_content).decode('utf-8') if b64_content else ''
    with open(full_path, 'w') as f:
        f.write(content)
    print(f'  \u2713 {rel_path}')

sys.path.insert(0, '/content')
print('\n\u2713 All modules written  |  /content added to sys.path')


In [ ]:
from multi_agent.mad_pipeline import run_mad
print('\u2713 MAD pipeline imported')

# Verify HF client is configured (model not loaded yet — lazy init)
import os
print(f'  Agent model : {os.environ["AGENT_MODEL"]}')
print(f'  Judge model : {os.environ["JUDGE_MODEL"]} ({os.environ["JUDGE_PROVIDER"]})')
print(f'  Embed model : {os.environ["EMBED_MODEL"]}')


In [ ]:
from rag.retriever import health_check

print('=== Qdrant Health Check ===')
result = health_check()
for k, v in result.items():
    print(f'  {k}: {v}')

if result['status'] == 'ok':
    print(f'\n\u2713 Qdrant connected  |  {result["points_count"]:,} chunks available')
else:
    print('\n\u2717 Qdrant connection failed')


In [ ]:
import time

# ── msq_013 — Flagship GDPR hallucination test ────────────────────────────
# LLMs frequently hallucinate that GDPR Art 32 mandates AES-256 / TLS 1.2.
# Reality: Art 32 says "appropriate technical and organisational measures" only.
# Expected MAD result: HARD_BLOCK, both claims v=0.0

QUERY = (
    'What specific encryption standards does GDPR Article 32 '
    'require organisations to implement?'
)

LLM_ANSWER = (
    'GDPR Article 32 requires organisations to implement AES-256 encryption '
    'for data at rest and TLS 1.2 or higher for data in transit. These are '
    'mandatory encryption standards specified by the regulation to ensure '
    'appropriate security of personal data.'
)

print(f'Query     : {QUERY}')
print(f'LLM answer: {LLM_ANSWER[:100]}...')
print()
print('Running MAD pipeline  (first run loads models — subsequent runs use cache)...')
print()

t0  = time.time()
out = run_mad(query=QUERY, llm_answer=LLM_ANSWER)
elapsed = time.time() - t0
print(f'\n\u23f1  Total time: {elapsed:.1f}s')


In [ ]:
print('\n' + '='*72)
print('  MAD RESULTS')
print('='*72)
print(f'  Routing             : {out.routing_decision}')
print(f'  Aggregate confidence: {out.aggregate_confidence:.4f}')
print()

print('  Judge verdicts:')
icons = {1.0: '\u2705', 0.5: '\u26a0\ufe0f ', 0.0: '\u274c'}
for jv in out.judge_verdicts:
    mat = '\u26a0 material' if jv.is_material else '  context '
    print(f'    {icons.get(jv.score,"?")} [{mat}] v={jv.score}  "{jv.claim_text[:65]}"')
    print(f'         {jv.reasoning[:130]}')

print()
if out.correction_signal:
    print('  Correction signal (sent to LLM on retry):')
    print(f'    {out.correction_signal[:300]}')
else:
    print('  Correction signal: null (all claims verified)')

print()
expected = 'HARD_BLOCK'
status   = '\u2705 PASS' if out.routing_decision == expected else f'\u274c FAIL (expected {expected})'
print(f'  Test result: {status}')
print('='*72)


## Full debate transcript (optional)
Expand to see every Agent A → Agent B → revision step.


In [ ]:
print(out.debate_transcript)


## Run your own query
Change `MY_QUERY` and `MY_LLM_ANSWER` below and re-run.


In [ ]:
MY_QUERY = 'Does HIPAA require covered entities to encrypt ePHI at rest?'

MY_LLM_ANSWER = (
    'Yes, HIPAA mandates that all covered entities must encrypt ePHI at rest '
    'using AES-256. Failure to implement encryption is a HIPAA violation subject '
    'to civil monetary penalties up to $50,000 per violation.'
)

my_result = run_mad(query=MY_QUERY, llm_answer=MY_LLM_ANSWER)
print(f'Routing: {my_result.routing_decision}  |  Score: {my_result.aggregate_confidence:.4f}')
if my_result.correction_signal:
    print(f'Correction: {my_result.correction_signal[:200]}')
